# NSS Computer Science Student Satisfaction Analysis — code-aligned revision

This notebook preserves the predeclared fixed holdout and runs repeated cross-validation only within the 80% development partition. It also records the descriptive checks and dimension-rank validation cited in the dissertation.


## 1. Imports and configuration

In [1]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

from pathlib import Path
import sys
import numpy as np
import pandas as pd

VERSION_DIR = Path.cwd()
PROJECT_DIR = VERSION_DIR.parent
LOCAL_DEPENDENCIES = VERSION_DIR / ".python_packages"
PROJECT_DEPENDENCIES = PROJECT_DIR / "version0.1" / ".python_packages"
DEPENDENCY_DIR = LOCAL_DEPENDENCIES if LOCAL_DEPENDENCIES.exists() else PROJECT_DEPENDENCIES
sys.path.insert(0, str(DEPENDENCY_DIR))

from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.calibration import CalibratedClassifierCV
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    cohen_kappa_score,
    mean_absolute_error,
    brier_score_loss,
    log_loss,
)
import statsmodels.api as sm
from statsmodels.miscmodels.ordinal_model import OrderedModel
from scipy.stats import chi2
from xgboost import XGBClassifier

RANDOM_STATE = 42
TEST_SIZE = 0.20
ORDINAL_MAXITER = 1000
MIN_GROUP_N_FOR_GAP = 100
CV_SPLITS = 5
CV_REPEATS = 5
CALIBRATION_BINS = 10

DATA_PATH = PROJECT_DIR / "dataset" / "dataset.xlsx"
OUTPUT_DIR = VERSION_DIR / "consistency_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 2. Load data


In [2]:
df_raw = pd.read_excel(DATA_PATH, sheet_name="Dataset")
print("Raw dataset shape:", df_raw.shape)
display(df_raw.head(3))

Raw dataset shape: (50915, 32)


,age_group,survey_year,sex,ethnicity_code,disability_reported,staff_explain_well,staff_make_subject_interesting,course_intellectually_stimulating,course_challenged_best_work,explore_ideas_in_depth,...,it_resources_support_learning,library_resources_support_learning,course_specific_resources_accessible,feel_part_of_community,opportunities_work_with_students,opportunities_provide_feedback,staff_value_student_views,feedback_acted_on_clear,students_union_represents_academic_interests,overall_satisfied_course_quality
0,U21,2018,F,.,0,5,4,5,5,4,...,5,5,5,5,5,5,5,4,4,5
1,U21,2018,F,.,0,4,4,5,5,3,...,4,4,5,4,3,5,4,4,3,5
2,U21,2018,F,.,0,4,4,4,3,4,...,4,4,5,4,4,5,5,4,4,4


## 3. Define NSS module groups and targets


In [3]:
target_col = "overall_satisfied_course_quality"
demographic_cols = [
    "age_group",
    "survey_year",
    "sex",
    "ethnicity_code",
    "disability_reported",
]

module_definitions = [
    {
        "nss_module": "Teaching on my course",
        "question_range": "Q1-Q4",
        "module_score_col": "module_teaching_on_my_course",
        "meaning_en": "Teaching quality, explanation, subject interest and intellectual challenge",
        "items": [
            "staff_explain_well",
            "staff_make_subject_interesting",
            "course_intellectually_stimulating",
            "course_challenged_best_work",
        ],
    },
    {
        "nss_module": "Learning opportunities",
        "question_range": "Q5-Q7",
        "module_score_col": "module_learning_opportunities",
        "meaning_en": "Opportunities to explore, combine and apply knowledge",
        "items": [
            "explore_ideas_in_depth",
            "bring_together_topics",
            "apply_what_learnt",
        ],
    },
    {
        "nss_module": "Assessment and feedback",
        "question_range": "Q8-Q11",
        "module_score_col": "module_assessment_and_feedback",
        "meaning_en": "Assessment criteria, fairness, timeliness and usefulness of feedback",
        "items": [
            "marking_criteria_clear",
            "assessment_fair",
            "feedback_timely",
            "feedback_helpful",
        ],
    },
    {
        "nss_module": "Academic support",
        "question_range": "Q12-Q14",
        "module_score_col": "module_academic_support",
        "meaning_en": "Contact with staff, advice and guidance",
        "items": [
            "contact_staff_when_needed",
            "sufficient_advice_guidance",
            "study_choices_advice_available",
        ],
    },
    {
        "nss_module": "Organisation and management",
        "question_range": "Q15-Q17",
        "module_score_col": "module_organisation_and_management",
        "meaning_en": "Course organisation, timetable and course management",
        "items": [
            "course_well_organised",
            "timetable_works_efficiently",
            "changes_communicated_effectively",
        ],
    },
    {
        "nss_module": "Learning resources",
        "question_range": "Q18-Q20",
        "module_score_col": "module_learning_resources",
        "meaning_en": "Library, IT and course-specific learning resources",
        "items": [
            "it_resources_support_learning",
            "library_resources_support_learning",
            "course_specific_resources_accessible",
        ],
    },
    {
        "nss_module": "Learning community",
        "question_range": "Q21-Q22",
        "module_score_col": "module_learning_community",
        "meaning_en": "Learning community, staff/student relations and belonging",
        "items": [
            "feel_part_of_community",
            "opportunities_work_with_students",
        ],
    },
    {
        "nss_module": "Student voice",
        "question_range": "Q23-Q26",
        "module_score_col": "module_student_voice",
        "meaning_en": "Student voice, feedback routes and whether student feedback is valued",
        "items": [
            "opportunities_provide_feedback",
            "staff_value_student_views",
            "feedback_acted_on_clear",
            "students_union_represents_academic_interests",
        ],
    },
]

module_definitions_df = pd.DataFrame([
    {
        "nss_module": module["nss_module"],
        "question_range": module["question_range"],
        "module_score_col": module["module_score_col"],
        "meaning_en": module["meaning_en"],
        "source_item": item,
    }
    for module in module_definitions
    for item in module["items"]
])
module_definitions_df.to_csv(OUTPUT_DIR / "module_score_feature_definitions_v0.1.csv", index=False)

module_score_cols = [m["module_score_col"] for m in module_definitions]
module_name_by_col = {m["module_score_col"]: m["nss_module"] for m in module_definitions}
module_range_by_col = {m["module_score_col"]: m["question_range"] for m in module_definitions}
all_item_cols = [item for module in module_definitions for item in module["items"]]

def add_module_scores(frame):
    scored = frame.copy()
    for module in module_definitions:
        scored[module["module_score_col"]] = scored[module["items"]].replace(0, np.nan).mean(axis=1)
    return scored


def clean_demographics(frame):
    cleaned = frame.copy()
    cleaned[demographic_cols] = cleaned[demographic_cols].astype(str).replace({".": "Unknown"})
    return cleaned

display(module_definitions_df.groupby(["nss_module", "question_range", "module_score_col"]).size().reset_index(name="n_items"))

,nss_module,question_range,module_score_col,n_items
0,Academic support,Q12-Q14,module_academic_support,3
1,Assessment and feedback,Q8-Q11,module_assessment_and_feedback,4
2,Learning community,Q21-Q22,module_learning_community,2
3,Learning opportunities,Q5-Q7,module_learning_opportunities,3
4,Learning resources,Q18-Q20,module_learning_resources,3
5,Organisation and management,Q15-Q17,module_organisation_and_management,3
6,Student voice,Q23-Q26,module_student_voice,4
7,Teaching on my course,Q1-Q4,module_teaching_on_my_course,4


## 4. Create module mean scores


In [4]:
df_modules = clean_demographics(add_module_scores(df_raw))
df_modules = df_modules[df_modules[target_col] != 0].copy()

y_ordinal = df_modules[target_col].astype(int)
y_binary = y_ordinal.isin([4, 5]).astype(int)

module_score_summary = (
    df_modules[module_score_cols]
    .agg(["count", "mean", "std", "min", "median", "max"])
    .T.rename_axis("module_score_col")
    .reset_index()
)
module_score_summary["nss_module"] = module_score_summary["module_score_col"].map(module_name_by_col)
module_score_summary["question_range"] = module_score_summary["module_score_col"].map(module_range_by_col)
module_score_summary = module_score_summary[[
    "nss_module", "question_range", "module_score_col", "count", "mean", "std", "min", "median", "max"
]]
module_score_summary.to_csv(OUTPUT_DIR / "module_score_summary_v0.1.csv", index=False)

ordinal_distribution = y_ordinal.value_counts().sort_index().rename_axis("Q27").reset_index(name="count")
ordinal_distribution["rate"] = ordinal_distribution["count"] / len(y_ordinal)
binary_distribution = y_binary.value_counts().sort_index().rename_axis("binary_satisfaction").reset_index(name="count")
binary_distribution["label"] = binary_distribution["binary_satisfaction"].map({0: "neutral_or_not_satisfied", 1: "satisfied"})
binary_distribution["rate"] = binary_distribution["count"] / len(y_binary)

print("Rows after removing Q27=0:", len(df_modules), "out of", len(df_raw))
print("Rows with any missing module score:", int(df_modules[module_score_cols].isna().any(axis=1).sum()))
display(module_score_summary)
display(ordinal_distribution)
display(binary_distribution)

Rows after removing Q27=0: 50633 out of 50915
Rows with any missing module score: 145


,nss_module,question_range,module_score_col,count,mean,std,min,median,max
0,Teaching on my course,Q1-Q4,module_teaching_on_my_course,50631.0,3.959521,0.789203,1.0,4.000000,5.0
1,Learning opportunities,Q5-Q7,module_learning_opportunities,50632.0,4.011090,0.826586,1.0,4.000000,5.0
2,Assessment and feedback,Q8-Q11,module_assessment_and_feedback,50628.0,3.713041,0.918711,1.0,3.750000,5.0
3,Academic support,Q12-Q14,module_academic_support,50611.0,3.965182,0.879704,1.0,4.000000,5.0
4,Organisation and management,Q15-Q17,module_organisation_and_management,50630.0,3.792600,0.916710,1.0,4.000000,5.0
5,Learning resources,Q18-Q20,module_learning_resources,50557.0,4.088514,0.861669,1.0,4.333333,5.0
6,Learning community,Q21-Q22,module_learning_community,50603.0,3.904472,0.940192,1.0,4.000000,5.0
7,Student voice,Q23-Q26,module_student_voice,50625.0,3.756128,0.878732,1.0,4.000000,5.0


,Q27,count,rate
0,1,2486,0.049098
1,2,4150,0.081962
2,3,6011,0.118717
3,4,22328,0.440977
4,5,15658,0.309245


,binary_satisfaction,count,label,rate
0,0,12647,neutral_or_not_satisfied,0.249778
1,1,37986,satisfied,0.750222


## 4a. Executable source and data-integrity audit

The supplied FOI workbook and the analysis workbook are compared value by value after aligning their column labels. The same cell records blankness, response-code validity, the treatment of code 0, incomplete dimension profiles and exact duplicate signatures. These checks make the data statements in the dissertation directly reproducible.


In [5]:
RAW_FOI_PATH = PROJECT_DIR / "dataset" / "FOI 585 - CompSci Final.xlsx"
foi_raw = pd.read_excel(RAW_FOI_PATH, sheet_name="Dataset")
foi_aligned = foi_raw.copy()
foi_aligned.columns = df_raw.columns

question_columns = all_item_cols + [target_col]
question_values = df_raw[question_columns]
valid_question_codes = question_values.isin([0, 1, 2, 3, 4, 5])

data_integrity_audit = pd.DataFrame([{
    "source_rows": len(foi_raw),
    "source_columns": foi_raw.shape[1],
    "analysis_rows": len(df_raw),
    "analysis_columns": df_raw.shape[1],
    "values_equal_after_column_rename": bool(foi_aligned.equals(df_raw)),
    "blank_cells_in_analysis_workbook": int(df_raw.isna().sum().sum()),
    "q1_q27_values_outside_0_to_5": int((~valid_question_codes).sum().sum()),
    "q27_code_zero_rows": int((df_raw[target_col] == 0).sum()),
    "analytic_rows_after_q27_exclusion": len(df_modules),
    "q1_q26_code_zero_responses_in_analytic_sample": int(
        (df_modules[all_item_cols] == 0).sum().sum()
    ),
    "analytic_rows_missing_at_least_one_dimension": int(
        df_modules[module_score_cols].isna().any(axis=1).sum()
    ),
    "exact_duplicate_rows_beyond_first_occurrence": int(
        df_modules.duplicated(keep="first").sum()
    ),
}])

annual_descriptive_summary = (
    df_modules.assign(satisfied=y_binary)
    .groupby("survey_year", as_index=False)
    .agg(
        n=(target_col, "size"),
        satisfied_rate=("satisfied", "mean"),
        mean_q27=(target_col, "mean"),
    )
)

data_integrity_audit.to_csv(
    OUTPUT_DIR / "data_integrity_audit_code_aligned.csv", index=False
)
annual_descriptive_summary.to_csv(
    OUTPUT_DIR / "annual_descriptive_summary_code_aligned.csv", index=False
)

display(data_integrity_audit)
display(annual_descriptive_summary.round(4))
del foi_raw, foi_aligned


,source_rows,source_columns,analysis_rows,analysis_columns,values_equal_after_column_rename,blank_cells_in_analysis_workbook,q1_q27_values_outside_0_to_5,q27_code_zero_rows,analytic_rows_after_q27_exclusion,q1_q26_code_zero_responses_in_analytic_sample,analytic_rows_missing_at_least_one_dimension,exact_duplicate_rows_beyond_first_occurrence
0,50915,32,50915,32,True,0,0,282,50633,10821,145,1454


,survey_year,n,satisfied_rate,mean_q27
0,2018,8904,0.7896,3.9867
1,2019,9425,0.7820,3.9649
2,2020,9887,0.7810,3.9565
3,2021,11063,0.6984,3.7452
4,2022,11354,0.7167,3.7875


## 5. Train/test split


In [6]:
train_idx, test_idx = train_test_split(
    df_modules.index,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_ordinal,
)

train_df = df_modules.loc[train_idx].copy()
test_df = df_modules.loc[test_idx].copy()
y_train_ord = y_ordinal.loc[train_idx]
y_test_ord = y_ordinal.loc[test_idx]
y_train_bin = y_binary.loc[train_idx]
y_test_bin = y_binary.loc[test_idx]

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))
display(y_train_ord.value_counts(normalize=True).sort_index().rename("train_q27_rate").to_frame())
display(y_test_ord.value_counts(normalize=True).sort_index().rename("test_q27_rate").to_frame())


Train rows: 40506
Test rows: 10127


,train_q27_rate
overall_satisfied_course_quality,
1,0.049104
2,0.081963
3,0.118723
4,0.440972
5,0.309238


,test_q27_rate
overall_satisfied_course_quality,
1,0.049077
2,0.081959
3,0.118693
4,0.440999
5,0.309272


## 6. Preprocessing, cached design matrices and evaluation helpers

All preprocessing operations are fitted inside the relevant training partition. For ordinal evaluation, the point prediction is the response category with the highest fitted probability (argmax). This rule is used consistently for accuracy, macro F1, QWK, MAE and within-one accuracy.

In [7]:
FEATURE_GROUPS = {
    "demographic": (demographic_cols, []),
    "module_scores": ([], module_score_cols),
    "module_plus_demographic": (demographic_cols, module_score_cols),
}


def make_preprocessor(feature_group, scale_numeric=True):
    categorical_cols, numeric_cols = FEATURE_GROUPS[feature_group]
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler() if scale_numeric else "passthrough"),
    ])
    transformers = [
        ("categorical", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False), categorical_cols),
        ("numeric", numeric_pipeline, numeric_cols),
    ]
    return ColumnTransformer(
        [transformer for transformer in transformers if transformer[2]],
        remainder="drop",
        verbose_feature_names_out=False,
    )


def build_design_matrices(train_frame, test_frame, feature_group, scale_numeric=True):
    preprocessor = make_preprocessor(feature_group, scale_numeric)
    X_train = np.asarray(preprocessor.fit_transform(train_frame), dtype=float)
    X_test = np.asarray(preprocessor.transform(test_frame), dtype=float)
    return X_train, X_test, preprocessor.get_feature_names_out().tolist(), preprocessor


def evaluate_ordinal_predictions(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "quadratic_weighted_kappa": cohen_kappa_score(y_true, y_pred, weights="quadratic"),
        "mean_absolute_error": mean_absolute_error(y_true, y_pred),
        "within_1": np.mean(np.abs(y_true - y_pred) <= 1),
    }


def evaluate_binary_predictions(y_true, y_pred, y_prob):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_satisfied": f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        "precision_satisfied": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "recall_satisfied": recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob),
    }


def tidy_metric_rows(rows):
    return pd.DataFrame(rows).sort_values(["task", "model", "feature_group"]).reset_index(drop=True)


def source_feature_name(feature):
    for base in module_score_cols + demographic_cols:
        if feature == base or feature.startswith(f"{base}_"):
            return base
    return feature


def feature_domain(feature):
    source = source_feature_name(feature)
    if source in module_score_cols:
        return "NSS module score"
    if source == "survey_year":
        return "Survey year"
    if source in demographic_cols:
        return "Student characteristics"
    return "Other"


scaled_design_matrices = {
    group: build_design_matrices(train_df, test_df, group, scale_numeric=True)
    for group in FEATURE_GROUPS
}
tree_design_matrices = {
    group: build_design_matrices(train_df, test_df, group, scale_numeric=False)
    for group in ["module_scores", "module_plus_demographic"]
}

## 6a. Dimension correlations and multicollinearity audit

Pairwise correlations use the analytic dimension scores. Variance inflation factors use the training-partition dimension matrix after the same training-fitted median imputation and standardisation as the primary ordinal model.


In [8]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

dimension_correlation_matrix = df_modules[module_score_cols].corr(method="pearson")
dimension_correlation_matrix.index = [
    module_name_by_col[column] for column in dimension_correlation_matrix.index
]
dimension_correlation_matrix.columns = [
    module_name_by_col[column] for column in dimension_correlation_matrix.columns
]

vif_matrix, _, vif_features, _ = scaled_design_matrices["module_scores"]
dimension_vif = pd.DataFrame({
    "feature": vif_features,
    "nss_module": [module_name_by_col[feature] for feature in vif_features],
    "variance_inflation_factor": [
        variance_inflation_factor(vif_matrix, position)
        for position in range(vif_matrix.shape[1])
    ],
}).sort_values("variance_inflation_factor").reset_index(drop=True)

dimension_correlation_matrix.to_csv(
    OUTPUT_DIR / "dimension_correlation_matrix_code_aligned.csv"
)
dimension_vif.to_csv(
    OUTPUT_DIR / "dimension_vif_code_aligned.csv", index=False
)

off_diagonal = dimension_correlation_matrix.where(
    ~np.eye(len(dimension_correlation_matrix), dtype=bool)
).stack()
print(
    f"Pairwise dimension correlations range from {off_diagonal.min():.4f} "
    f"to {off_diagonal.max():.4f}."
)
display(dimension_vif.round(4))


Pairwise dimension correlations range from 0.4242 to 0.7250.


,feature,nss_module,variance_inflation_factor
0,module_learning_resources,Learning resources,1.6070
1,module_learning_community,Learning community,2.0288
2,module_assessment_and_feedback,Assessment and feedback,2.1558
3,module_organisation_and_management,Organisation and management,2.3315
4,module_learning_opportunities,Learning opportunities,2.4421
5,module_academic_support,Academic support,2.4851
6,module_student_voice,Student voice,2.7020
7,module_teaching_on_my_course,Teaching on my course,2.7705


## 7. Baseline models


In [9]:
metric_rows = []

ord_majority_class = int(y_train_ord.mode().iloc[0])
ord_baseline_pred = np.full(len(y_test_ord), ord_majority_class, dtype=int)
metric_rows.append({
    "task": "ordinal_1_to_5",
    "model": "Baseline majority class",
    "feature_group": "none",
    "n_features": 0,
    **evaluate_ordinal_predictions(y_test_ord, ord_baseline_pred),
})

bin_majority_class = int(y_train_bin.mode().iloc[0])
bin_baseline_pred = np.full(len(y_test_bin), bin_majority_class, dtype=int)
bin_baseline_prob = np.full(len(y_test_bin), y_train_bin.mean())
metric_rows.append({
    "task": "binary_satisfied",
    "model": "Baseline majority class",
    "feature_group": "none",
    "n_features": 0,
    **evaluate_binary_predictions(y_test_bin, bin_baseline_pred, bin_baseline_prob),
})

print("Ordinal majority class:", ord_majority_class)
print("Binary majority class:", bin_majority_class)
display(tidy_metric_rows(metric_rows))

Ordinal majority class: 4
Binary majority class: 1


,task,model,feature_group,n_features,accuracy,macro_f1,weighted_f1,quadratic_weighted_kappa,mean_absolute_error,within_1,balanced_accuracy,f1_satisfied,precision_satisfied,recall_satisfied,roc_auc
0,binary_satisfied,Baseline majority class,none,0,0.750272,0.428660,NaN,NaN,NaN,NaN,0.5,0.85732,0.750272,1.0,0.5
1,ordinal_1_to_5,Baseline majority class,none,0,0.440999,0.122415,0.269924,0.0,0.739113,0.868964,NaN,NaN,NaN,NaN,NaN


## 8. Ordinal Logistic Regression with module scores


In [10]:
ordinal_artifacts = {}
for group, (X_train, X_test, feature_names, preprocessor) in scaled_design_matrices.items():
    model = OrderedModel(y_train_ord, X_train, distr="logit")
    result = model.fit(method="lbfgs", maxiter=ORDINAL_MAXITER, disp=False)
    pred_probs = result.model.predict(result.params, exog=X_test)
    pred = np.asarray(pred_probs).argmax(axis=1) + 1

    metric_rows.append({
        "task": "ordinal_1_to_5",
        "model": "Ordinal Logistic Regression",
        "feature_group": group,
        "n_features": X_train.shape[1],
        "converged": bool(result.mle_retvals.get("converged", False)),
        "aic": result.aic,
        "bic": result.bic,
        **evaluate_ordinal_predictions(y_test_ord, pred),
    })
    ordinal_artifacts[group] = {
        "result": result,
        "feature_names": feature_names,
        "preprocessor": preprocessor,
    }

ordinal_metric_table = tidy_metric_rows([row for row in metric_rows if row["task"] == "ordinal_1_to_5"])
display(ordinal_metric_table)

,task,model,feature_group,n_features,accuracy,macro_f1,weighted_f1,quadratic_weighted_kappa,mean_absolute_error,within_1,converged,aic,bic
0,ordinal_1_to_5,Baseline majority class,none,0,0.440999,0.122415,0.269924,0.000000,0.739113,0.868964,NaN,NaN,NaN
1,ordinal_1_to_5,Ordinal Logistic Regression,demographic,28,0.440999,0.122415,0.269924,0.000000,0.739113,0.868964,True,107348.039739,107623.534311
2,ordinal_1_to_5,Ordinal Logistic Regression,module_plus_demographic,36,0.662684,0.568251,0.650295,0.791410,0.377111,0.963563,True,64107.639841,64452.008057
3,ordinal_1_to_5,Ordinal Logistic Regression,module_scores,8,0.663770,0.567285,0.651260,0.792089,0.375827,0.963661,True,64191.705403,64295.015868


## 9. Binary Logistic Regression with module scores


In [11]:
binary_logistic_artifacts = {}
for group, (X_train, X_test, feature_names, preprocessor) in scaled_design_matrices.items():
    classifier = LogisticRegression(max_iter=3000, solver="lbfgs")
    classifier.fit(X_train, y_train_bin)
    pred = classifier.predict(X_test)
    prob = classifier.predict_proba(X_test)[:, 1]

    metric_rows.append({
        "task": "binary_satisfied",
        "model": "Binary Logistic Regression",
        "feature_group": group,
        "n_features": X_train.shape[1],
        **evaluate_binary_predictions(y_test_bin, pred, prob),
    })
    binary_logistic_artifacts[group] = {
        "classifier": classifier,
        "feature_names": feature_names,
        "preprocessor": preprocessor,
    }

binary_logistic_metric_table = tidy_metric_rows([
    row for row in metric_rows
    if row["task"] == "binary_satisfied"
    and row["model"] in ["Baseline majority class", "Binary Logistic Regression"]
])
display(binary_logistic_metric_table)

,task,model,feature_group,n_features,accuracy,balanced_accuracy,macro_f1,f1_satisfied,precision_satisfied,recall_satisfied,roc_auc
0,binary_satisfied,Baseline majority class,none,0,0.750272,0.500000,0.428660,0.857320,0.750272,1.000000,0.500000
1,binary_satisfied,Binary Logistic Regression,demographic,28,0.750272,0.500000,0.428660,0.857320,0.750272,1.000000,0.560827
2,binary_satisfied,Binary Logistic Regression,module_plus_demographic,36,0.890590,0.825787,0.844887,0.929083,0.904311,0.955251,0.938557
3,binary_satisfied,Binary Logistic Regression,module_scores,8,0.889602,0.825129,0.843725,0.928398,0.904192,0.953935,0.938412


## 10. Binary comparison models with grouped dimension predictors

The fixed benchmark specifications favour regularisation and computational tractability. They are not tuned against the holdout set and do not represent each algorithm's maximum achievable performance.

In [12]:
binary_model_artifacts = {}
for group in ["module_scores", "module_plus_demographic"]:
    model_specs = [
        ("Naive Bayes", GaussianNB(), scaled_design_matrices[group]),
        ("Decision Tree", DecisionTreeClassifier(
            max_depth=4, min_samples_leaf=300, random_state=RANDOM_STATE
        ), tree_design_matrices[group]),
        ("Random Forest", RandomForestClassifier(
            n_estimators=200,
            min_samples_leaf=50,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ), tree_design_matrices[group]),
        ("XGBoost", XGBClassifier(
            n_estimators=250,
            max_depth=3,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            n_jobs=-1,
            tree_method="hist",
        ), tree_design_matrices[group]),
    ]

    for model_name, classifier, design in model_specs:
        X_train, X_test, feature_names, _ = design
        classifier.fit(X_train, y_train_bin)
        pred = classifier.predict(X_test)
        prob = classifier.predict_proba(X_test)[:, 1]

        metric_rows.append({
            "task": "binary_satisfied",
            "model": model_name,
            "feature_group": group,
            "n_features": X_train.shape[1],
            **evaluate_binary_predictions(y_test_bin, pred, prob),
        })
        binary_model_artifacts[(model_name, group)] = {
            "classifier": classifier,
            "feature_names": feature_names,
        }

binary_all_metric_table = tidy_metric_rows([row for row in metric_rows if row["task"] == "binary_satisfied"])
display(binary_all_metric_table)

,task,model,feature_group,n_features,accuracy,balanced_accuracy,macro_f1,f1_satisfied,precision_satisfied,recall_satisfied,roc_auc
0,binary_satisfied,Baseline majority class,none,0,0.750272,0.500000,0.428660,0.857320,0.750272,1.000000,0.500000
1,binary_satisfied,Binary Logistic Regression,demographic,28,0.750272,0.500000,0.428660,0.857320,0.750272,1.000000,0.560827
2,binary_satisfied,Binary Logistic Regression,module_plus_demographic,36,0.890590,0.825787,0.844887,0.929083,0.904311,0.955251,0.938557
3,binary_satisfied,Binary Logistic Regression,module_scores,8,0.889602,0.825129,0.843725,0.928398,0.904192,0.953935,0.938412
4,binary_satisfied,Decision Tree,module_plus_demographic,36,0.873013,0.800752,0.819308,0.917817,0.892050,0.945117,0.917975
5,binary_satisfied,Decision Tree,module_scores,8,0.873013,0.800752,0.819308,0.917817,0.892050,0.945117,0.917975
6,binary_satisfied,Naive Bayes,module_plus_demographic,36,0.859583,0.836516,0.820889,0.904139,0.926755,0.882601,0.915400
7,binary_satisfied,Naive Bayes,module_scores,8,0.874395,0.847838,0.837311,0.914985,0.929522,0.900895,0.933171
8,binary_satisfied,Random Forest,module_plus_demographic,36,0.854449,0.859870,0.823351,0.897468,0.951756,0.849039,0.937566
9,binary_satisfied,Random Forest,module_scores,8,0.857510,0.860459,0.826134,0.899993,0.950520,0.854567,0.937871


## 11. Model comparison output


In [13]:
combined_results = tidy_metric_rows(metric_rows)
combined_results.to_csv(OUTPUT_DIR / "model_results_v0.1.csv", index=False)

display(combined_results)


,task,model,feature_group,n_features,accuracy,macro_f1,weighted_f1,quadratic_weighted_kappa,mean_absolute_error,within_1,balanced_accuracy,f1_satisfied,precision_satisfied,recall_satisfied,roc_auc,converged,aic,bic
0,binary_satisfied,Baseline majority class,none,0,0.750272,0.428660,NaN,NaN,NaN,NaN,0.500000,0.857320,0.750272,1.000000,0.500000,NaN,NaN,NaN
1,binary_satisfied,Binary Logistic Regression,demographic,28,0.750272,0.428660,NaN,NaN,NaN,NaN,0.500000,0.857320,0.750272,1.000000,0.560827,NaN,NaN,NaN
2,binary_satisfied,Binary Logistic Regression,module_plus_demographic,36,0.890590,0.844887,NaN,NaN,NaN,NaN,0.825787,0.929083,0.904311,0.955251,0.938557,NaN,NaN,NaN
3,binary_satisfied,Binary Logistic Regression,module_scores,8,0.889602,0.843725,NaN,NaN,NaN,NaN,0.825129,0.928398,0.904192,0.953935,0.938412,NaN,NaN,NaN
4,binary_satisfied,Decision Tree,module_plus_demographic,36,0.873013,0.819308,NaN,NaN,NaN,NaN,0.800752,0.917817,0.892050,0.945117,0.917975,NaN,NaN,NaN
5,binary_satisfied,Decision Tree,module_scores,8,0.873013,0.819308,NaN,NaN,NaN,NaN,0.800752,0.917817,0.892050,0.945117,0.917975,NaN,NaN,NaN
6,binary_satisfied,Naive Bayes,module_plus_demographic,36,0.859583,0.820889,NaN,NaN,NaN,NaN,0.836516,0.904139,0.926755,0.882601,0.915400,NaN,NaN,NaN
7,binary_satisfied,Naive Bayes,module_scores,8,0.874395,0.837311,NaN,NaN,NaN,NaN,0.847838,0.914985,0.929522,0.900895,0.933171,NaN,NaN,NaN
8,binary_satisfied,Random Forest,module_plus_demographic,36,0.854449,0.823351,NaN,NaN,NaN,NaN,0.859870,0.897468,0.951756,0.849039,0.937566,NaN,NaN,NaN
9,binary_satisfied,Random Forest,module_scores,8,0.857510,0.826134,NaN,NaN,NaN,NaN,0.860459,0.899993,0.950520,0.854567,0.937871,NaN,NaN,NaN


## 12. Model-specific dimension rankings

Intrinsic importance measures have different statistical meanings across model families. The analysis therefore retains each model's dimension ranking separately and summarises rank stability. It does not average coefficient magnitudes, impurity importance and gain importance into a common contribution share.

In [14]:
importance_tables = []

# Ordinal logistic coefficient magnitudes.
for group, artifact in ordinal_artifacts.items():
    feature_names = artifact["feature_names"]
    params = np.asarray(artifact["result"].params[:len(feature_names)], dtype=float)
    importance_tables.append(pd.DataFrame({
        "model": "Ordinal Logistic Regression",
        "feature_group": group,
        "feature": feature_names,
        "importance_type": "absolute standardised coefficient",
        "importance": np.abs(params),
        "coefficient": params,
        "odds_ratio": np.exp(params),
    }))

# Binary logistic coefficient magnitudes.
for group, artifact in binary_logistic_artifacts.items():
    classifier = artifact["classifier"]
    params = classifier.coef_[0]
    importance_tables.append(pd.DataFrame({
        "model": "Binary Logistic Regression",
        "feature_group": group,
        "feature": artifact["feature_names"],
        "importance_type": "absolute standardised coefficient",
        "importance": np.abs(params),
        "coefficient": params,
        "odds_ratio": np.exp(params),
    }))

# Tree-based intrinsic importance values.
for model_name in ["Decision Tree", "Random Forest", "XGBoost"]:
    for group in ["module_scores", "module_plus_demographic"]:
        artifact = binary_model_artifacts[(model_name, group)]
        importance_tables.append(pd.DataFrame({
            "model": model_name,
            "feature_group": group,
            "feature": artifact["feature_names"],
            "importance_type": "impurity importance" if model_name != "XGBoost" else "gain importance",
            "importance": artifact["classifier"].feature_importances_,
            "coefficient": np.nan,
            "odds_ratio": np.nan,
        }))

feature_importance_df = pd.concat(importance_tables, ignore_index=True)
feature_importance_df["source_feature"] = feature_importance_df["feature"].map(source_feature_name)
feature_importance_df["domain"] = feature_importance_df["feature"].map(feature_domain)
feature_importance_df["nss_module"] = feature_importance_df["source_feature"].map(module_name_by_col)
feature_importance_df["question_range"] = feature_importance_df["source_feature"].map(module_range_by_col)
feature_importance_df["model_id"] = feature_importance_df["model"] + " | " + feature_importance_df["feature_group"]
feature_importance_df["overall_feature_rank"] = feature_importance_df.groupby("model_id")["importance"].rank(
    ascending=False, method="min"
)

dimension_mask = feature_importance_df["domain"] == "NSS module score"
feature_importance_df.loc[dimension_mask, "dimension_rank"] = (
    feature_importance_df.loc[dimension_mask]
    .groupby("model_id")["importance"]
    .rank(ascending=False, method="min")
)
feature_importance_df = feature_importance_df.sort_values(
    ["model", "feature_group", "importance"], ascending=[True, True, False]
).reset_index(drop=True)
dimension_mask = feature_importance_df["domain"] == "NSS module score"
feature_importance_df.to_csv(
    OUTPUT_DIR / "module_feature_importance_v0.1.csv", index=False
)

module_model_rankings = feature_importance_df.loc[
    dimension_mask,
    [
        "model", "feature_group", "model_id", "nss_module", "question_range",
        "source_feature", "importance_type", "importance", "coefficient",
        "odds_ratio", "dimension_rank",
    ],
].copy()
module_model_rankings["dimension_rank"] = module_model_rankings["dimension_rank"].astype(int)
module_model_rankings = module_model_rankings.sort_values(
    ["model_id", "dimension_rank", "nss_module"]
).reset_index(drop=True)
module_model_rankings.to_csv(
    OUTPUT_DIR / "module_model_specific_rankings_v0.1.csv", index=False
)

module_rank_summary = (
    module_model_rankings
    .groupby(["nss_module", "question_range", "source_feature"], as_index=False)
    .agg(
        model_configurations=("model_id", "nunique"),
        median_dimension_rank=("dimension_rank", "median"),
        mean_dimension_rank=("dimension_rank", "mean"),
        best_dimension_rank=("dimension_rank", "min"),
        worst_dimension_rank=("dimension_rank", "max"),
        first_place_count=("dimension_rank", lambda ranks: int((ranks == 1).sum())),
        top_two_count=("dimension_rank", lambda ranks: int((ranks <= 2).sum())),
    )
    .sort_values(
        ["median_dimension_rank", "mean_dimension_rank", "first_place_count", "nss_module"],
        ascending=[True, True, False, True],
    )
    .reset_index(drop=True)
)
module_rank_summary.insert(0, "summary_rank", np.arange(1, len(module_rank_summary) + 1))
module_rank_summary.to_csv(
    OUTPUT_DIR / "module_rank_summary_v0.1.csv", index=False
)

# Compatibility output: this file now contains rank summaries, not averaged importance shares.
module_rank_summary.to_csv(
    OUTPUT_DIR / "module_factor_ranking_v0.1.csv", index=False
)

rank_matrix = module_model_rankings.pivot(
    index="nss_module", columns="model_id", values="dimension_rank"
)
rank_agreement = rank_matrix.corr(method="spearman")
rank_agreement.to_csv(
    OUTPUT_DIR / "module_rank_agreement_spearman_v0.1.csv"
)

module_coefficients = feature_importance_df.loc[
    dimension_mask
    & feature_importance_df["model"].isin([
        "Ordinal Logistic Regression", "Binary Logistic Regression"
    ])
].copy()
module_coefficients = module_coefficients.sort_values(
    ["model", "feature_group", "dimension_rank", "nss_module"]
).reset_index(drop=True)
module_coefficients.to_csv(
    OUTPUT_DIR / "module_score_coefficients_v0.1.csv", index=False
)

print("Model-specific dimension-rank summary")
display(module_rank_summary)
print("Dimension ranks by model configuration")
display(rank_matrix)
print("Spearman agreement among model-specific dimension rankings")
display(rank_agreement.round(3))

Model-specific dimension-rank summary


,summary_rank,nss_module,question_range,source_feature,model_configurations,median_dimension_rank,mean_dimension_rank,best_dimension_rank,worst_dimension_rank,first_place_count,top_two_count
0,1,Teaching on my course,Q1-Q4,module_teaching_on_my_course,10,1.0,1.0,1,1,10,10
1,2,Organisation and management,Q15-Q17,module_organisation_and_management,10,2.0,2.0,2,2,0,10
2,3,Learning opportunities,Q5-Q7,module_learning_opportunities,10,3.0,3.9,3,7,0,0
3,4,Academic support,Q12-Q14,module_academic_support,10,4.0,4.2,3,5,0,0
4,5,Student voice,Q23-Q26,module_student_voice,10,4.0,4.3,3,5,0,0
5,6,Learning community,Q21-Q22,module_learning_community,10,6.0,6.4,6,7,0,0
6,7,Learning resources,Q18-Q20,module_learning_resources,10,6.5,6.5,5,8,0,0
7,8,Assessment and feedback,Q8-Q11,module_assessment_and_feedback,10,8.0,7.7,6,8,0,0


Dimension ranks by model configuration


model_id,Binary Logistic Regression | module_plus_demographic,Binary Logistic Regression | module_scores,Decision Tree | module_plus_demographic,Decision Tree | module_scores,Ordinal Logistic Regression | module_plus_demographic,Ordinal Logistic Regression | module_scores,Random Forest | module_plus_demographic,Random Forest | module_scores,XGBoost | module_plus_demographic,XGBoost | module_scores
nss_module,,,,,,,,,,
Academic support,4,4,3,3,5,5,5,5,4,4
Assessment and feedback,8,8,8,8,8,8,7,6,8,8
Learning community,7,7,6,6,6,7,6,7,6,6
Learning opportunities,3,3,7,7,3,3,4,3,3,3
Learning resources,6,6,5,5,7,6,8,8,7,7
Organisation and management,2,2,2,2,2,2,2,2,2,2
Student voice,5,5,4,4,4,4,3,4,5,5
Teaching on my course,1,1,1,1,1,1,1,1,1,1


Spearman agreement among model-specific dimension rankings


model_id,Binary Logistic Regression | module_plus_demographic,Binary Logistic Regression | module_scores,Decision Tree | module_plus_demographic,Decision Tree | module_scores,Ordinal Logistic Regression | module_plus_demographic,Ordinal Logistic Regression | module_scores,Random Forest | module_plus_demographic,Random Forest | module_scores,XGBoost | module_plus_demographic,XGBoost | module_scores
model_id,,,,,,,,,,
Binary Logistic Regression | module_plus_demographic,1.000,1.000,0.762,0.762,0.952,0.976,0.857,0.881,0.976,0.976
Binary Logistic Regression | module_scores,1.000,1.000,0.762,0.762,0.952,0.976,0.857,0.881,0.976,0.976
Decision Tree | module_plus_demographic,0.762,0.762,1.000,1.000,0.714,0.738,0.714,0.595,0.738,0.738
Decision Tree | module_scores,0.762,0.762,1.000,1.000,0.714,0.738,0.714,0.595,0.738,0.738
Ordinal Logistic Regression | module_plus_demographic,0.952,0.952,0.714,0.714,1.000,0.976,0.952,0.929,0.976,0.976
Ordinal Logistic Regression | module_scores,0.976,0.976,0.738,0.738,0.976,1.000,0.905,0.905,0.952,0.952
Random Forest | module_plus_demographic,0.857,0.857,0.714,0.714,0.952,0.905,1.000,0.952,0.905,0.905
Random Forest | module_scores,0.881,0.881,0.595,0.595,0.929,0.905,0.952,1.000,0.905,0.905
XGBoost | module_plus_demographic,0.976,0.976,0.738,0.738,0.976,0.952,0.905,0.905,1.000,1.000


## 13. Compact interpretation


In [15]:
print("Compact interpretation: dimension-score models")

ord_table = combined_results[combined_results["task"] == "ordinal_1_to_5"].copy()
bin_table = combined_results[combined_results["task"] == "binary_satisfied"].copy()

best_ord = ord_table.sort_values("quadratic_weighted_kappa", ascending=False).iloc[0]
best_bin_auc = bin_table.sort_values("roc_auc", ascending=False).iloc[0]
best_bin_f1 = bin_table.sort_values("macro_f1", ascending=False).iloc[0]

print(
    f"Best ordinal model by QWK: {best_ord['model']} ({best_ord['feature_group']}), "
    f"QWK={best_ord['quadratic_weighted_kappa']:.3f}, accuracy={best_ord['accuracy']:.3f}."
)
print(
    f"Best binary model by ROC-AUC: {best_bin_auc['model']} ({best_bin_auc['feature_group']}), "
    f"ROC-AUC={best_bin_auc['roc_auc']:.3f}, accuracy={best_bin_auc['accuracy']:.3f}."
)
print(
    f"Best binary model by macro F1: {best_bin_f1['model']} ({best_bin_f1['feature_group']}), "
    f"macro F1={best_bin_f1['macro_f1']:.3f}, accuracy={best_bin_f1['accuracy']:.3f}."
)

print("\nModel-specific NSS dimension ranks:")
display(module_rank_summary[[
    "summary_rank", "nss_module", "question_range", "median_dimension_rank",
    "mean_dimension_rank", "first_place_count", "top_two_count",
]])

print("\nInterpretation rule:")
print(
    "Intrinsic importance magnitudes are interpreted within each model family. "
    "Cross-model evidence uses rank stability and leave-one-dimension-out ablation, not an averaged contribution share."
)

Compact interpretation: dimension-score models
Best ordinal model by QWK: Ordinal Logistic Regression (module_scores), QWK=0.792, accuracy=0.664.
Best binary model by ROC-AUC: XGBoost (module_plus_demographic), ROC-AUC=0.940, accuracy=0.889.
Best binary model by macro F1: Binary Logistic Regression (module_plus_demographic), macro F1=0.845, accuracy=0.891.

Model-specific NSS dimension ranks:


,summary_rank,nss_module,question_range,median_dimension_rank,mean_dimension_rank,first_place_count,top_two_count
0,1,Teaching on my course,Q1-Q4,1.0,1.0,10,10
1,2,Organisation and management,Q15-Q17,2.0,2.0,0,10
2,3,Learning opportunities,Q5-Q7,3.0,3.9,0,0
3,4,Academic support,Q12-Q14,4.0,4.2,0,0
4,5,Student voice,Q23-Q26,4.0,4.3,0,0
5,6,Learning community,Q21-Q22,6.0,6.4,0,0
6,7,Learning resources,Q18-Q20,6.5,6.5,0,0
7,8,Assessment and feedback,Q8-Q11,8.0,7.7,0,0



Interpretation rule:
Intrinsic importance magnitudes are interpreted within each model family. Cross-model evidence uses rank stability and leave-one-dimension-out ablation, not an averaged contribution share.


## 14. Notes for dissertation write-up

The individual Q1-Q26 variables construct the eight questionnaire-section mean scores; they are not entered separately into the Q27 models. Substantive interpretation should use the primary ordinal coefficients, threshold-specific coefficients, model-specific dimension ranks and leave-one-dimension-out losses. Intrinsic importance magnitudes from different model families do not share a common contribution scale. Prediction terminology is reserved for held-out model performance, and none of these quantities establishes a causal effect.

## 15. Student Characteristics and Q1-Q26 Responses

This section integrates the v0.05 analysis. It evaluates whether age group, survey year, sex, ethnicity code and disability status are associated with Q1-Q26 and the eight NSS module scores.


### 15.1 Prepare item and theme metadata

In [16]:
short_labels = [
    "Staff explain well",
    "Staff make subject interesting",
    "Course intellectually stimulating",
    "Course challenged best work",
    "Explore ideas in depth",
    "Bring together topics",
    "Apply what learnt",
    "Marking criteria clear",
    "Assessment fair",
    "Feedback timely",
    "Feedback helpful",
    "Contact staff when needed",
    "Sufficient advice and guidance",
    "Study choices advice available",
    "Course well organised",
    "Timetable works efficiently",
    "Changes communicated effectively",
    "IT resources support learning",
    "Library resources support learning",
    "Course-specific resources accessible",
    "Feel part of community",
    "Opportunities to work with students",
    "Opportunities to provide feedback",
    "Staff value student views",
    "Feedback acted on clear",
    "Students union represents academic interests",
]

question_metadata = pd.DataFrame({
    "question": [f"Q{i}" for i in range(1, 27)],
    "feature": all_item_cols,
    "nss_module": [
        module["nss_module"]
        for module in module_definitions
        for _ in module["items"]
    ],
    "short_label": short_labels,
})

module_metadata = pd.DataFrame([
    {
        "nss_module": module["nss_module"],
        "question_range": module["question_range"],
        "n_items": len(module["items"]),
        "module_score_col": module["module_score_col"],
    }
    for module in module_definitions
])

df = clean_demographics(add_module_scores(df_raw))
module_metadata.to_csv(OUTPUT_DIR / "module_metadata_v0.1.csv", index=False)
question_metadata.to_csv(OUTPUT_DIR / "question_metadata_v0.1.csv", index=False)
display(module_metadata)

,nss_module,question_range,n_items,module_score_col
0,Teaching on my course,Q1-Q4,4,module_teaching_on_my_course
1,Learning opportunities,Q5-Q7,3,module_learning_opportunities
2,Assessment and feedback,Q8-Q11,4,module_assessment_and_feedback
3,Academic support,Q12-Q14,3,module_academic_support
4,Organisation and management,Q15-Q17,3,module_organisation_and_management
5,Learning resources,Q18-Q20,3,module_learning_resources
6,Learning community,Q21-Q22,2,module_learning_community
7,Student voice,Q23-Q26,4,module_student_voice


### 15.2 Group-difference helpers

In [17]:
def eta_squared_for_group(frame, value_col, group_col):
    data = frame[[value_col, group_col]].dropna()
    overall_mean = data[value_col].mean()
    total_ss = ((data[value_col] - overall_mean) ** 2).sum()
    grouped = data.groupby(group_col)[value_col]
    between_ss = (((grouped.mean() - overall_mean) ** 2) * grouped.size()).sum()
    return float(between_ss / total_ss)


def group_mean_table(frame, metadata, level_name):
    group_tables = []
    effect_rows = []

    for _, meta in metadata.iterrows():
        value_col = meta["feature"] if level_name == "question" else meta["module_score_col"]
        label = meta["short_label"] if level_name == "question" else meta["nss_module"]
        question_range = meta["question"] if level_name == "question" else meta["question_range"]
        data = frame[demographic_cols + [value_col]].copy()
        if level_name == "question":
            data[value_col] = data[value_col].replace(0, np.nan)

        for characteristic in demographic_cols:
            group_stats = (
                data[[characteristic, value_col]]
                .dropna()
                .groupby(characteristic)[value_col]
                .agg(n="count", mean="mean", std="std", median="median")
                .reset_index()
                .rename(columns={characteristic: "category"})
            )
            group_stats = group_stats.assign(
                level=level_name,
                question_or_range=question_range,
                feature=value_col,
                label=label,
                nss_module=meta["nss_module"],
                student_characteristic=characteristic,
            )
            group_tables.append(group_stats)

            eligible = group_stats[group_stats["n"] >= MIN_GROUP_N_FOR_GAP]
            highest = eligible.loc[eligible["mean"].idxmax()]
            lowest = eligible.loc[eligible["mean"].idxmin()]
            effect_rows.append({
                "level": level_name,
                "question_or_range": question_range,
                "feature": value_col,
                "label": label,
                "nss_module": meta["nss_module"],
                "student_characteristic": characteristic,
                "eta_squared": eta_squared_for_group(data, value_col, characteristic),
                "max_minus_min_mean_gap_min_n_100": float(highest["mean"] - lowest["mean"]),
                "highest_mean_category": highest["category"],
                "highest_mean": float(highest["mean"]),
                "lowest_mean_category": lowest["category"],
                "lowest_mean": float(lowest["mean"]),
                "n_categories": int(group_stats["category"].nunique()),
            })

    return pd.concat(group_tables, ignore_index=True), pd.DataFrame(effect_rows)

### 15.3 Group differences for Q1-Q26 and theme scores

In [18]:
question_group_means, question_effect_sizes = group_mean_table(df, question_metadata, level_name="question")
module_group_means, module_effect_sizes = group_mean_table(df, module_metadata, level_name="module")

question_group_means.to_csv(OUTPUT_DIR / "question_group_means_by_student_characteristic_v0.1.csv", index=False)
question_effect_sizes.to_csv(OUTPUT_DIR / "question_characteristic_effect_sizes_v0.1.csv", index=False)
module_group_means.to_csv(OUTPUT_DIR / "module_group_means_by_student_characteristic_v0.1.csv", index=False)
module_effect_sizes.to_csv(OUTPUT_DIR / "module_characteristic_effect_sizes_v0.1.csv", index=False)

print("Top Q1-Q26 group differences by eta squared")
display(question_effect_sizes.sort_values("eta_squared", ascending=False).head(20))
print("Top module group differences by eta squared")
display(module_effect_sizes.sort_values("eta_squared", ascending=False).head(20))

Top Q1-Q26 group differences by eta squared


,level,question_or_range,feature,label,nss_module,student_characteristic,eta_squared,max_minus_min_mean_gap_min_n_100,highest_mean_category,highest_mean,lowest_mean_category,lowest_mean,n_categories
96,question,Q20,course_specific_resources_accessible,Course-specific resources accessible,Learning resources,survey_year,0.015459,0.317591,2020,4.273362,2021,3.955772,5
106,question,Q22,opportunities_work_with_students,Opportunities to work with students,Learning community,survey_year,0.013072,0.238956,2018,4.288246,2021,4.049290,5
101,question,Q21,feel_part_of_community,Feel part of community,Learning community,survey_year,0.012239,0.303373,2018,3.767183,2021,3.463810,5
86,question,Q18,it_resources_support_learning,IT resources support learning,Learning resources,survey_year,0.011389,0.285293,2020,4.173948,2021,3.888655,5
91,question,Q19,library_resources_support_learning,Library resources support learning,Learning resources,survey_year,0.009589,0.254894,2020,4.089355,2021,3.834461,5
43,question,Q9,assessment_fair,Assessment fair,Assessment and feedback,ethnicity_code,0.009577,0.365918,41,3.973958,50,3.608040,20
128,question,Q26,students_union_represents_academic_interests,Students union represents academic interests,Student voice,ethnicity_code,0.008048,0.363908,22,3.689511,13,3.325603,20
121,question,Q25,feedback_acted_on_clear,Feedback acted on clear,Student voice,survey_year,0.007542,0.230604,2018,3.640634,2021,3.410030,5
58,question,Q12,contact_staff_when_needed,Contact staff when needed,Academic support,ethnicity_code,0.007405,0.425690,Unknown,4.340426,33,3.914736,20
123,question,Q25,feedback_acted_on_clear,Feedback acted on clear,Student voice,ethnicity_code,0.006545,0.529657,34,3.787041,13,3.257384,20


Top module group differences by eta squared


,level,question_or_range,feature,label,nss_module,student_characteristic,eta_squared,max_minus_min_mean_gap_min_n_100,highest_mean_category,highest_mean,lowest_mean_category,lowest_mean,n_categories
26,module,Q18-Q20,module_learning_resources,Learning resources,Learning resources,survey_year,0.017336,0.290741,2020,4.178944,2021,3.888203,5
31,module,Q21-Q22,module_learning_community,Learning community,Learning community,survey_year,0.016358,0.272343,2018,4.027312,2021,3.754969,5
16,module,Q12-Q14,module_academic_support,Academic support,Academic support,survey_year,0.006588,0.154548,2019,4.028391,2021,3.873844,5
18,module,Q12-Q14,module_academic_support,Academic support,Academic support,ethnicity_code,0.005674,0.290515,41,4.066494,33,3.775979,20
36,module,Q23-Q26,module_student_voice,Student voice,Student voice,survey_year,0.005673,0.140872,2018,3.821659,2022,3.680787,5
10,module,Q8-Q11,module_assessment_and_feedback,Assessment and feedback,Assessment and feedback,age_group,0.005556,0.275297,31+,3.954792,U21,3.679495,4
13,module,Q8-Q11,module_assessment_and_feedback,Assessment and feedback,Assessment and feedback,ethnicity_code,0.005091,0.263689,41,3.800781,49,3.537092,20
1,module,Q1-Q4,module_teaching_on_my_course,Teaching on my course,Teaching on my course,survey_year,0.003738,0.117995,2018,4.006138,2021,3.888143,5
38,module,Q23-Q26,module_student_voice,Student voice,Student voice,ethnicity_code,0.003536,0.324583,Unknown,3.934697,49,3.610113,20
21,module,Q15-Q17,module_organisation_and_management,Organisation and management,Organisation and management,survey_year,0.003471,0.129912,2018,3.854613,2021,3.724701,5


### 15.4 Ordinal association models for Q1-Q26

In [19]:
question_ordinal_rows = []
question_coefficient_rows = []

for _, meta in question_metadata.iterrows():
    question = meta["question"]
    target = meta["feature"]
    model_df = df.loc[df[target] != 0, demographic_cols + [target]].copy()
    y = model_df[target].astype(int)
    row_base = {
        "question": question,
        "target_feature": target,
        "short_label": meta["short_label"],
        "nss_module": meta["nss_module"],
        "n_rows": len(model_df),
        "n_classes": int(y.nunique()),
    }

    question_train_idx, question_test_idx = train_test_split(
        model_df.index,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=y,
    )
    y_train = y.loc[question_train_idx]
    y_test = y.loc[question_test_idx]

    baseline_class = int(y_train.mode().iloc[0])
    baseline_pred = np.full(len(y_test), baseline_class, dtype=int)
    question_ordinal_rows.append({
        **row_base,
        "model": "Baseline majority class",
        "n_features": 0,
        **evaluate_ordinal_predictions(y_test, baseline_pred),
    })

    preprocessor = ColumnTransformer(
        [("categorical", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False), demographic_cols)],
        remainder="drop",
        verbose_feature_names_out=False,
    )
    X_train = np.asarray(preprocessor.fit_transform(model_df.loc[question_train_idx]), dtype=float)
    X_test = np.asarray(preprocessor.transform(model_df.loc[question_test_idx]), dtype=float)
    feature_names = preprocessor.get_feature_names_out().tolist()

    result = OrderedModel(y_train, X_train, distr="logit").fit(
        method="lbfgs",
        maxiter=ORDINAL_MAXITER,
        disp=False,
    )
    pred_probs = result.model.predict(result.params, exog=X_test)
    pred = np.asarray(pred_probs).argmax(axis=1) + 1
    question_ordinal_rows.append({
        **row_base,
        "model": "Ordinal Logistic Regression",
        "n_features": X_train.shape[1],
        "converged": bool(result.mle_retvals.get("converged", False)),
        "aic": result.aic,
        "bic": result.bic,
        **evaluate_ordinal_predictions(y_test, pred),
    })

    params = np.asarray(result.params[:len(feature_names)], dtype=float)
    pvalues = np.asarray(result.pvalues[:len(feature_names)], dtype=float)
    for feature_name, coefficient, p_value in zip(feature_names, params, pvalues):
        question_coefficient_rows.append({
            "question": question,
            "target_feature": target,
            "short_label": meta["short_label"],
            "nss_module": meta["nss_module"],
            "encoded_feature": feature_name,
            "student_characteristic": source_feature_name(feature_name),
            "coefficient": coefficient,
            "absolute_coefficient": abs(coefficient),
            "odds_ratio": float(np.exp(coefficient)),
            "p_value": p_value,
        })

question_ordinal_results = pd.DataFrame(question_ordinal_rows).drop(columns="weighted_f1")
question_ordinal_coefficients = pd.DataFrame(question_coefficient_rows)
question_ordinal_results.to_csv(OUTPUT_DIR / "demographic_to_questions_ordinal_results_v0.1.csv", index=False)
question_ordinal_coefficients.to_csv(OUTPUT_DIR / "demographic_to_questions_ordinal_coefficients_v0.1.csv", index=False)

question_ordinal_models = question_ordinal_results[
    question_ordinal_results["model"] == "Ordinal Logistic Regression"
].copy()
print("Ordinal demographic models completed:", len(question_ordinal_models), "of", len(question_metadata))
display(question_ordinal_results.head(12))
display(question_ordinal_models.sort_values("quadratic_weighted_kappa", ascending=False).head(10))

Ordinal demographic models completed: 26 of 26


,question,target_feature,short_label,nss_module,n_rows,n_classes,model,n_features,accuracy,macro_f1,quadratic_weighted_kappa,mean_absolute_error,within_1,converged,aic,bic
0,Q1,staff_explain_well,Staff explain well,Teaching on my course,50877,5,Baseline majority class,0,0.582252,0.147196,0.000000,0.513070,0.923644,NaN,NaN,NaN
1,Q1,staff_explain_well,Staff explain well,Teaching on my course,50877,5,Ordinal Logistic Regression,28,0.582252,0.147196,0.000000,0.513070,0.923644,True,91839.507062,92115.155316
2,Q2,staff_make_subject_interesting,Staff make subject interesting,Teaching on my course,50879,5,Baseline majority class,0,0.493612,0.132193,0.000000,0.636989,0.896325,NaN,NaN,NaN
3,Q2,staff_make_subject_interesting,Staff make subject interesting,Teaching on my course,50879,5,Ordinal Logistic Regression,28,0.493612,0.132193,0.000000,0.636989,0.896325,True,103703.902685,103979.552512
4,Q3,course_intellectually_stimulating,Course intellectually stimulating,Teaching on my course,50855,5,Baseline majority class,0,0.436830,0.121609,0.000000,0.669649,0.919084,NaN,NaN,NaN
5,Q3,course_intellectually_stimulating,Course intellectually stimulating,Teaching on my course,50855,5,Ordinal Logistic Regression,28,0.436339,0.135695,0.010112,0.675155,0.916724,True,100820.131295,101095.766181
6,Q4,course_challenged_best_work,Course challenged best work,Teaching on my course,50877,5,Baseline majority class,0,0.388561,0.111932,0.000000,0.754520,0.892885,NaN,NaN,NaN
7,Q4,course_challenged_best_work,Course challenged best work,Teaching on my course,50877,5,Ordinal Logistic Regression,28,0.391411,0.168358,0.024111,0.810633,0.860161,True,107022.888012,107298.536266
8,Q5,explore_ideas_in_depth,Explore ideas in depth,Learning opportunities,50849,5,Baseline majority class,0,0.424287,0.119158,0.000000,0.707866,0.898427,NaN,NaN,NaN
9,Q5,explore_ideas_in_depth,Explore ideas in depth,Learning opportunities,50849,5,Ordinal Logistic Regression,28,0.425762,0.130785,0.005809,0.712389,0.894690,True,105177.670617,105453.301570


,question,target_feature,short_label,nss_module,n_rows,n_classes,model,n_features,accuracy,macro_f1,quadratic_weighted_kappa,mean_absolute_error,within_1,converged,aic,bic
43,Q22,opportunities_work_with_students,Opportunities to work with students,Learning community,50632,5,Ordinal Logistic Regression,28,0.437444,0.189554,0.095079,0.718969,0.890886,True,95116.137909,95391.631692
39,Q20,course_specific_resources_accessible,Course-specific resources accessible,Learning resources,50356,5,Ordinal Logistic Regression,28,0.447776,0.177056,0.076790,0.761319,0.865866,True,95371.702045,95647.020754
35,Q18,it_resources_support_learning,IT resources support learning,Learning resources,50410,5,Ordinal Logistic Regression,28,0.423825,0.175954,0.068110,0.842293,0.831581,True,101749.090486,102024.444128
51,Q26,students_union_represents_academic_interests,Students union represents academic interests,Student voice,47792,5,Ordinal Logistic Regression,28,0.339366,0.164295,0.066161,0.889214,0.796213,True,110467.117292,110740.763829
37,Q19,library_resources_support_learning,Library resources support learning,Learning resources,48350,5,Ordinal Logistic Regression,28,0.400931,0.172727,0.065249,0.876939,0.804860,True,100262.068011,100536.086505
13,Q7,apply_what_learnt,Apply what learnt,Learning opportunities,50847,5,Ordinal Logistic Regression,28,0.425172,0.178839,0.050210,0.729007,0.892429,True,100676.505020,100952.134399
45,Q23,opportunities_provide_feedback,Opportunities to provide feedback,Student voice,50636,5,Ordinal Logistic Regression,28,0.429601,0.188420,0.046100,0.755727,0.875790,True,97726.193054,98001.689207
23,Q12,contact_staff_when_needed,Contact staff when needed,Academic support,50691,5,Ordinal Logistic Regression,28,0.457244,0.170849,0.029149,0.789526,0.847125,True,97159.958303,97435.489196
31,Q16,timetable_works_efficiently,Timetable works efficiently,Organisation and management,50807,5,Ordinal Logistic Regression,28,0.399528,0.180570,0.028014,0.855639,0.837630,True,107466.889308,107742.493503
47,Q24,staff_value_student_views,Staff value student views,Student voice,50495,5,Ordinal Logistic Regression,28,0.377067,0.149483,0.024944,0.832360,0.854738,True,111066.885643,111342.293196


### 15.5 Student-characteristic summary

In [20]:
question_effect_summary = (
    question_effect_sizes
    .groupby("student_characteristic", as_index=False)
    .agg(
        mean_eta_squared_q1_q26=("eta_squared", "mean"),
        max_eta_squared_q1_q26=("eta_squared", "max"),
        mean_group_gap_q1_q26=("max_minus_min_mean_gap_min_n_100", "mean"),
        max_group_gap_q1_q26=("max_minus_min_mean_gap_min_n_100", "max"),
    )
    .sort_values("mean_eta_squared_q1_q26", ascending=False)
)

module_effect_summary = (
    module_effect_sizes
    .groupby("student_characteristic", as_index=False)
    .agg(
        mean_eta_squared_modules=("eta_squared", "mean"),
        max_eta_squared_modules=("eta_squared", "max"),
        mean_group_gap_modules=("max_minus_min_mean_gap_min_n_100", "mean"),
        max_group_gap_modules=("max_minus_min_mean_gap_min_n_100", "max"),
    )
    .sort_values("mean_eta_squared_modules", ascending=False)
)

coefficient_summary = (
    question_ordinal_coefficients
    .groupby("student_characteristic", as_index=False)
    .agg(
        mean_abs_ordinal_coefficient=("absolute_coefficient", "mean"),
        median_abs_ordinal_coefficient=("absolute_coefficient", "median"),
        max_abs_ordinal_coefficient=("absolute_coefficient", "max"),
        n_encoded_levels=("encoded_feature", "nunique"),
    )
    .sort_values("mean_abs_ordinal_coefficient", ascending=False)
)

student_characteristic_summary = (
    question_effect_summary
    .merge(module_effect_summary, on="student_characteristic", how="outer")
    .merge(coefficient_summary, on="student_characteristic", how="left")
    .sort_values("mean_eta_squared_q1_q26", ascending=False)
)
student_characteristic_summary.to_csv(OUTPUT_DIR / "student_characteristic_q1_q26_summary_v0.1.csv", index=False)

baseline = question_ordinal_results[question_ordinal_results["model"] == "Baseline majority class"].copy()
pred_summary = question_ordinal_models[["question", "target_feature", "short_label", "nss_module", "accuracy", "macro_f1", "quadratic_weighted_kappa", "within_1"]].merge(
    baseline[["question", "accuracy", "quadratic_weighted_kappa"]].rename(columns={"accuracy": "baseline_accuracy", "quadratic_weighted_kappa": "baseline_qwk"}),
    on="question",
    how="left",
)
pred_summary["accuracy_gain_vs_baseline"] = pred_summary["accuracy"] - pred_summary["baseline_accuracy"]
pred_summary["qwk_gain_vs_baseline"] = pred_summary["quadratic_weighted_kappa"] - pred_summary["baseline_qwk"]
pred_summary.to_csv(OUTPUT_DIR / "demographic_to_questions_predictive_summary_v0.1.csv", index=False)

print("Student characteristic summary")
display(student_characteristic_summary)
print("Predictive summary: demographic-only models for Q1-Q26")
display(pred_summary.sort_values("quadratic_weighted_kappa", ascending=False).head(15))


Student characteristic summary


,student_characteristic,mean_eta_squared_q1_q26,max_eta_squared_q1_q26,mean_group_gap_q1_q26,max_group_gap_q1_q26,mean_eta_squared_modules,max_eta_squared_modules,mean_group_gap_modules,max_group_gap_modules,mean_abs_ordinal_coefficient,median_abs_ordinal_coefficient,max_abs_ordinal_coefficient,n_encoded_levels
4,survey_year,0.004782,0.015459,0.161706,0.317591,0.007328,0.017336,0.167872,0.290741,0.133134,0.099221,0.492508,4
2,ethnicity_code,0.003654,0.009577,0.342195,0.529657,0.003231,0.005674,0.293822,0.411859,0.162678,0.147456,0.589283,19
0,age_group,0.001464,0.004447,0.161545,0.318767,0.001579,0.005556,0.145080,0.275297,0.126433,0.113054,0.304948,3
1,disability_reported,0.000786,0.002626,0.063016,0.139918,0.000850,0.002774,0.055829,0.117167,0.098184,0.084772,0.242313,1
3,sex,0.000303,0.001310,0.038933,0.099609,0.000261,0.001073,0.028874,0.076790,0.066047,0.058641,0.171441,1


Predictive summary: demographic-only models for Q1-Q26


,question,target_feature,short_label,nss_module,accuracy,macro_f1,quadratic_weighted_kappa,within_1,baseline_accuracy,baseline_qwk,accuracy_gain_vs_baseline,qwk_gain_vs_baseline
21,Q22,opportunities_work_with_students,Opportunities to work with students,Learning community,0.437444,0.189554,0.095079,0.890886,0.429940,0.0,0.007505,0.095079
19,Q20,course_specific_resources_accessible,Course-specific resources accessible,Learning resources,0.447776,0.177056,0.076790,0.865866,0.450953,0.0,-0.003177,0.076790
17,Q18,it_resources_support_learning,IT resources support learning,Learning resources,0.423825,0.175954,0.068110,0.831581,0.424717,0.0,-0.000893,0.068110
25,Q26,students_union_represents_academic_interests,Students union represents academic interests,Student voice,0.339366,0.164295,0.066161,0.796213,0.322942,0.0,0.016424,0.066161
18,Q19,library_resources_support_learning,Library resources support learning,Learning resources,0.400931,0.172727,0.065249,0.804860,0.387280,0.0,0.013650,0.065249
6,Q7,apply_what_learnt,Apply what learnt,Learning opportunities,0.425172,0.178839,0.050210,0.892429,0.420551,0.0,0.004621,0.050210
22,Q23,opportunities_provide_feedback,Opportunities to provide feedback,Student voice,0.429601,0.188420,0.046100,0.875790,0.415581,0.0,0.014021,0.046100
11,Q12,contact_staff_when_needed,Contact staff when needed,Academic support,0.457244,0.170849,0.029149,0.847125,0.452411,0.0,0.004833,0.029149
15,Q16,timetable_works_efficiently,Timetable works efficiently,Organisation and management,0.399528,0.180570,0.028014,0.837630,0.385456,0.0,0.014072,0.028014
23,Q24,staff_value_student_views,Staff value student views,Student voice,0.377067,0.149483,0.024944,0.854738,0.377265,0.0,-0.000198,0.024944


### 15.6 Compact interpretation

In [21]:
print("v0.1 compact interpretation")

print()
print("1. Average effect sizes across Q1-Q26")
display(student_characteristic_summary[[
    "student_characteristic", "mean_eta_squared_q1_q26", "max_eta_squared_q1_q26", "mean_group_gap_q1_q26"
]])

print()
print("2. Average effect sizes across the 8 NSS questionnaire sections")
display(student_characteristic_summary[[
    "student_characteristic", "mean_eta_squared_modules", "max_eta_squared_modules", "mean_group_gap_modules"
]].sort_values("mean_eta_squared_modules", ascending=False))

print()
print("3. Demographic-only prediction of Q1-Q26")
print("Mean QWK:", round(pred_summary["quadratic_weighted_kappa"].mean(), 3))
print("Mean accuracy gain over majority baseline:", round(pred_summary["accuracy_gain_vs_baseline"].mean(), 3))
display(pred_summary.sort_values("quadratic_weighted_kappa", ascending=False).head(10))

print()
print("Interpretation rule:")
print("These outputs describe associations between student characteristics and NSS experience responses. They do not establish causal effects.")


v0.1 compact interpretation

1. Average effect sizes across Q1-Q26


,student_characteristic,mean_eta_squared_q1_q26,max_eta_squared_q1_q26,mean_group_gap_q1_q26
4,survey_year,0.004782,0.015459,0.161706
2,ethnicity_code,0.003654,0.009577,0.342195
0,age_group,0.001464,0.004447,0.161545
1,disability_reported,0.000786,0.002626,0.063016
3,sex,0.000303,0.001310,0.038933



2. Average effect sizes across the 8 NSS questionnaire sections


,student_characteristic,mean_eta_squared_modules,max_eta_squared_modules,mean_group_gap_modules
4,survey_year,0.007328,0.017336,0.167872
2,ethnicity_code,0.003231,0.005674,0.293822
0,age_group,0.001579,0.005556,0.145080
1,disability_reported,0.000850,0.002774,0.055829
3,sex,0.000261,0.001073,0.028874



3. Demographic-only prediction of Q1-Q26
Mean QWK: 0.025
Mean accuracy gain over majority baseline: 0.003


,question,target_feature,short_label,nss_module,accuracy,macro_f1,quadratic_weighted_kappa,within_1,baseline_accuracy,baseline_qwk,accuracy_gain_vs_baseline,qwk_gain_vs_baseline
21,Q22,opportunities_work_with_students,Opportunities to work with students,Learning community,0.437444,0.189554,0.095079,0.890886,0.429940,0.0,0.007505,0.095079
19,Q20,course_specific_resources_accessible,Course-specific resources accessible,Learning resources,0.447776,0.177056,0.076790,0.865866,0.450953,0.0,-0.003177,0.076790
17,Q18,it_resources_support_learning,IT resources support learning,Learning resources,0.423825,0.175954,0.068110,0.831581,0.424717,0.0,-0.000893,0.068110
25,Q26,students_union_represents_academic_interests,Students union represents academic interests,Student voice,0.339366,0.164295,0.066161,0.796213,0.322942,0.0,0.016424,0.066161
18,Q19,library_resources_support_learning,Library resources support learning,Learning resources,0.400931,0.172727,0.065249,0.804860,0.387280,0.0,0.013650,0.065249
6,Q7,apply_what_learnt,Apply what learnt,Learning opportunities,0.425172,0.178839,0.050210,0.892429,0.420551,0.0,0.004621,0.050210
22,Q23,opportunities_provide_feedback,Opportunities to provide feedback,Student voice,0.429601,0.188420,0.046100,0.875790,0.415581,0.0,0.014021,0.046100
11,Q12,contact_staff_when_needed,Contact staff when needed,Academic support,0.457244,0.170849,0.029149,0.847125,0.452411,0.0,0.004833,0.029149
15,Q16,timetable_works_efficiently,Timetable works efficiently,Organisation and management,0.399528,0.180570,0.028014,0.837630,0.385456,0.0,0.014072,0.028014
23,Q24,staff_value_student_views,Staff value student views,Student voice,0.377067,0.149483,0.024944,0.854738,0.377265,0.0,-0.000198,0.024944



Interpretation rule:
These outputs describe associations between student characteristics and NSS experience responses. They do not establish causal effects.


### 15.7 Notes for dissertation write-up

The conclusion should combine effect sizes with demographic-only model performance. Low eta-squared values and QWK scores support the narrower claim that the five recorded student characteristics have limited associations with, and limited predictive power for, individual NSS experience responses. This does not imply that unrecorded background or institutional factors are unimportant.

## 16. Internal consistency of the eight NSS dimensions

Cronbach's alpha and standardised alpha assess whether the questions grouped into each official NSS dimension move together consistently. For the two-item Learning community dimension, the Spearman-Brown coefficient is reported because alpha is especially sensitive to the number of items.

In [22]:
def cronbach_statistics(item_frame):
    complete = item_frame.replace(0, np.nan).dropna()
    n_items = complete.shape[1]
    item_variance = complete.var(axis=0, ddof=1).sum()
    total_variance = complete.sum(axis=1).var(ddof=1)
    alpha = n_items / (n_items - 1) * (1 - item_variance / total_variance)

    correlation = complete.corr()
    upper_triangle = correlation.to_numpy()[np.triu_indices(n_items, k=1)]
    mean_inter_item_r = upper_triangle.mean()
    standardised_alpha = n_items * mean_inter_item_r / (1 + (n_items - 1) * mean_inter_item_r)
    spearman_brown = 2 * mean_inter_item_r / (1 + mean_inter_item_r) if n_items == 2 else np.nan
    return complete, alpha, standardised_alpha, mean_inter_item_r, spearman_brown


reliability_rows = []
item_diagnostic_rows = []

for module in module_definitions:
    items = module["items"]
    complete, alpha, standardised_alpha, mean_r, spearman_brown = cronbach_statistics(df_modules[items])
    reliability_rows.append({
        "nss_module": module["nss_module"],
        "question_range": module["question_range"],
        "n_items": len(items),
        "complete_case_n": len(complete),
        "cronbach_alpha": alpha,
        "standardised_alpha": standardised_alpha,
        "mean_inter_item_correlation": mean_r,
        "spearman_brown_two_item": spearman_brown,
    })

    for item in items:
        other_items = [other for other in items if other != item]
        corrected_item_total = complete[item].corr(complete[other_items].sum(axis=1))
        alpha_if_deleted = np.nan
        if len(other_items) >= 2:
            _, alpha_if_deleted, _, _, _ = cronbach_statistics(complete[other_items])
        item_diagnostic_rows.append({
            "nss_module": module["nss_module"],
            "question_range": module["question_range"],
            "item": item,
            "corrected_item_total_correlation": corrected_item_total,
            "alpha_if_item_deleted": alpha_if_deleted,
        })

module_reliability = pd.DataFrame(reliability_rows)
module_item_diagnostics = pd.DataFrame(item_diagnostic_rows)
module_reliability.to_csv(OUTPUT_DIR / "module_internal_consistency_v0.1.csv", index=False)
module_item_diagnostics.to_csv(OUTPUT_DIR / "module_item_diagnostics_v0.1.csv", index=False)

display(module_reliability.round(3))

,nss_module,question_range,n_items,complete_case_n,cronbach_alpha,standardised_alpha,mean_inter_item_correlation,spearman_brown_two_item
0,Teaching on my course,Q1-Q4,4,50484,0.837,0.839,0.566,NaN
1,Learning opportunities,Q5-Q7,3,50417,0.781,0.781,0.543,NaN
2,Assessment and feedback,Q8-Q11,4,50388,0.815,0.816,0.526,NaN
3,Academic support,Q12-Q14,3,49305,0.815,0.814,0.594,NaN
4,Organisation and management,Q15-Q17,3,49990,0.739,0.738,0.485,NaN
5,Learning resources,Q18-Q20,3,47518,0.786,0.786,0.551,NaN
6,Learning community,Q21-Q22,2,50121,0.685,0.698,0.536,0.698
7,Student voice,Q23-Q26,4,47320,0.810,0.810,0.517,NaN


## 17. Proportional-odds diagnostic for the primary ordinal summary model

The diagnostic tests whether each dimension has a common slope across all four cumulative response thresholds. The proportional-odds model is retained as a parsimonious summary model rather than an exact threshold-invariant representation. Threshold-specific coefficients show where the common-slope approximation changes in magnitude.

In [23]:
def brant_style_wald_test(X, y, feature_names):
    X_with_intercept = sm.add_constant(np.asarray(X, dtype=float), has_constant="add")
    y_array = np.asarray(y, dtype=int)
    thresholds = sorted(np.unique(y_array))[:-1]
    parameter_rows = []
    parameter_vectors = []
    inverse_information = []
    score_matrices = []

    for threshold in thresholds:
        binary_target = (y_array > threshold).astype(int)
        fitted = sm.Logit(binary_target, X_with_intercept).fit(method="lbfgs", maxiter=1000, disp=False)
        probability = fitted.predict(X_with_intercept)
        weights = probability * (1 - probability)
        information = X_with_intercept.T @ (weights[:, None] * X_with_intercept)
        inverse_information.append(np.linalg.pinv(information))
        score_matrices.append(X_with_intercept * (binary_target - probability)[:, None])
        parameter_vectors.append(np.asarray(fitted.params))

        for feature, coefficient in zip(feature_names, fitted.params[1:]):
            parameter_rows.append({
                "threshold": f"Q27 > {threshold}",
                "feature": feature,
                "nss_module": module_name_by_col.get(feature, feature),
                "coefficient": coefficient,
                "odds_ratio": np.exp(coefficient),
            })

    block_rows = []
    for left in range(len(thresholds)):
        block_row = []
        for right in range(len(thresholds)):
            score_covariance = score_matrices[left].T @ score_matrices[right]
            block_row.append(
                inverse_information[left] @ score_covariance @ inverse_information[right].T
            )
        block_rows.append(block_row)

    stacked_parameters = np.concatenate(parameter_vectors)
    stacked_covariance = np.block(block_rows)
    parameters_per_model = X_with_intercept.shape[1]

    def wald_result(feature_indices):
        constraints = []
        for threshold_index in range(1, len(thresholds)):
            for feature_index in feature_indices:
                row = np.zeros(len(stacked_parameters))
                row[feature_index] = -1
                row[threshold_index * parameters_per_model + feature_index] = 1
                constraints.append(row)
        constraint_matrix = np.vstack(constraints)
        differences = constraint_matrix @ stacked_parameters
        covariance = constraint_matrix @ stacked_covariance @ constraint_matrix.T
        statistic = float(differences.T @ np.linalg.pinv(covariance) @ differences)
        degrees_of_freedom = int(np.linalg.matrix_rank(covariance))
        return statistic, degrees_of_freedom, float(chi2.sf(statistic, degrees_of_freedom))

    diagnostic_rows = []
    global_statistic, global_df, global_p = wald_result(range(1, parameters_per_model))
    diagnostic_rows.append({
        "feature": "__GLOBAL__",
        "nss_module": "All eight NSS dimensions",
        "wald_chi_square": global_statistic,
        "degrees_of_freedom": global_df,
        "p_value": global_p,
        "proportional_odds_supported_at_0_05": global_p >= 0.05,
    })

    for feature_index, feature in enumerate(feature_names, start=1):
        statistic, degrees_of_freedom, p_value = wald_result([feature_index])
        diagnostic_rows.append({
            "feature": feature,
            "nss_module": module_name_by_col.get(feature, feature),
            "wald_chi_square": statistic,
            "degrees_of_freedom": degrees_of_freedom,
            "p_value": p_value,
            "proportional_odds_supported_at_0_05": p_value >= 0.05,
        })

    coefficient_table = pd.DataFrame(parameter_rows)
    coefficient_spread = (
        coefficient_table.groupby(["feature", "nss_module"], as_index=False)["coefficient"]
        .agg(coefficient_min="min", coefficient_max="max", coefficient_sd="std")
    )
    coefficient_spread["coefficient_range"] = (
        coefficient_spread["coefficient_max"] - coefficient_spread["coefficient_min"]
    )
    diagnostics = pd.DataFrame(diagnostic_rows)
    diagnostics = diagnostics.merge(coefficient_spread, on=["feature", "nss_module"], how="left")
    return diagnostics, coefficient_table


X_train_modules, _, module_feature_names, _ = scaled_design_matrices["module_scores"]
proportional_odds_diagnostic, threshold_coefficients = brant_style_wald_test(
    X_train_modules, y_train_ord, module_feature_names
)
proportional_odds_diagnostic.to_csv(
    OUTPUT_DIR / "proportional_odds_diagnostic_v0.1.csv", index=False
)
threshold_coefficients.to_csv(
    OUTPUT_DIR / "proportional_odds_threshold_coefficients_v0.1.csv", index=False
)

display(proportional_odds_diagnostic.round(4))
display(threshold_coefficients.pivot(index="nss_module", columns="threshold", values="coefficient").round(3))

,feature,nss_module,wald_chi_square,degrees_of_freedom,p_value,proportional_odds_supported_at_0_05,coefficient_min,coefficient_max,coefficient_sd,coefficient_range
0,__GLOBAL__,All eight NSS dimensions,312.8844,24,0.0000,False,NaN,NaN,NaN,NaN
1,module_teaching_on_my_course,Teaching on my course,18.9646,3,0.0003,False,1.1430,1.3479,0.0943,0.2049
2,module_learning_opportunities,Learning opportunities,6.2120,3,0.1017,True,0.3287,0.4391,0.0456,0.1103
3,module_assessment_and_feedback,Assessment and feedback,15.3730,3,0.0015,False,0.0819,0.2678,0.0783,0.1859
4,module_academic_support,Academic support,14.0968,3,0.0028,False,0.2142,0.3553,0.0628,0.1411
5,module_organisation_and_management,Organisation and management,8.8409,3,0.0315,False,0.5310,0.6614,0.0566,0.1305
6,module_learning_resources,Learning resources,61.8329,3,0.0000,False,0.1129,0.3746,0.1210,0.2617
7,module_learning_community,Learning community,11.6286,3,0.0088,False,0.1468,0.3046,0.0645,0.1579
8,module_student_voice,Student voice,30.5502,3,0.0000,False,0.2861,0.5436,0.1114,0.2574


threshold,Q27 > 1,Q27 > 2,Q27 > 3,Q27 > 4
nss_module,,,,
Academic support,0.243,0.214,0.301,0.355
Assessment and feedback,0.082,0.158,0.204,0.268
Learning community,0.147,0.228,0.231,0.305
Learning opportunities,0.329,0.388,0.371,0.439
Learning resources,0.113,0.143,0.273,0.375
Organisation and management,0.531,0.661,0.637,0.609
Student voice,0.544,0.342,0.286,0.420
Teaching on my course,1.143,1.218,1.319,1.348


## 18. Five-by-five repeated stratified cross-validation

Twenty-five train-validation splits are generated within the fixed 80% development partition. The untouched 20% holdout is not used in these folds. All preprocessing is refitted inside each split. Ordinal models compare dimension-only and combined specifications; binary models are evaluated with both feature groups. Dimension-only ordinal coefficients and within-fold ranks are also retained so any claim about ranking stability is supported by an explicit output.

In [24]:
cv_splitter = RepeatedStratifiedKFold(
    n_splits=CV_SPLITS,
    n_repeats=CV_REPEATS,
    random_state=RANDOM_STATE,
)
cv_rows = []
cv_dimension_coefficient_rows = []

for split_number, (cv_train_pos, cv_valid_pos) in enumerate(
    cv_splitter.split(train_df, y_train_ord), start=1
):
    repeat_number = (split_number - 1) // CV_SPLITS + 1
    fold_number = (split_number - 1) % CV_SPLITS + 1
    cv_train = train_df.iloc[cv_train_pos]
    cv_valid = train_df.iloc[cv_valid_pos]
    cv_y_train_ord = y_train_ord.iloc[cv_train_pos]
    cv_y_valid_ord = y_train_ord.iloc[cv_valid_pos]
    cv_y_train_bin = y_train_bin.iloc[cv_train_pos]
    cv_y_valid_bin = y_train_bin.iloc[cv_valid_pos]

    ordinal_majority = int(cv_y_train_ord.mode().iloc[0])
    ordinal_baseline = np.full(len(cv_y_valid_ord), ordinal_majority, dtype=int)
    cv_rows.append({
        "repeat": repeat_number,
        "fold": fold_number,
        "task": "ordinal_1_to_5",
        "model": "Baseline majority class",
        "feature_group": "none",
        **evaluate_ordinal_predictions(cv_y_valid_ord, ordinal_baseline),
    })

    binary_majority = int(cv_y_train_bin.mode().iloc[0])
    binary_baseline = np.full(len(cv_y_valid_bin), binary_majority, dtype=int)
    binary_baseline_probability = np.full(len(cv_y_valid_bin), cv_y_train_bin.mean())
    cv_rows.append({
        "repeat": repeat_number,
        "fold": fold_number,
        "task": "binary_satisfied",
        "model": "Baseline majority class",
        "feature_group": "none",
        **evaluate_binary_predictions(
            cv_y_valid_bin, binary_baseline, binary_baseline_probability
        ),
        "brier_score": brier_score_loss(cv_y_valid_bin, binary_baseline_probability),
    })

    for group in ["module_scores", "module_plus_demographic"]:
        scaled_train, scaled_valid, _, _ = build_design_matrices(
            cv_train, cv_valid, group, scale_numeric=True
        )
        tree_train, tree_valid, _, _ = build_design_matrices(
            cv_train, cv_valid, group, scale_numeric=False
        )

        ordinal_model = OrderedModel(cv_y_train_ord, scaled_train, distr="logit")
        ordinal_result = ordinal_model.fit(
            method="lbfgs", maxiter=ORDINAL_MAXITER, disp=False
        )
        ordinal_probability = ordinal_result.model.predict(
            ordinal_result.params, exog=scaled_valid
        )
        ordinal_prediction = np.asarray(ordinal_probability).argmax(axis=1) + 1
        if group == "module_scores":
            fold_coefficients = np.asarray(
                ordinal_result.params[:len(module_score_cols)], dtype=float
            )
            fold_ranks = (
                pd.Series(np.abs(fold_coefficients))
                .rank(ascending=False, method="min").astype(int).to_numpy()
            )
            for feature, coefficient, rank in zip(
                module_score_cols, fold_coefficients, fold_ranks
            ):
                cv_dimension_coefficient_rows.append({
                    "repeat": repeat_number,
                    "fold": fold_number,
                    "feature": feature,
                    "nss_module": module_name_by_col[feature],
                    "coefficient": coefficient,
                    "dimension_rank": int(rank),
                })
        cv_rows.append({
            "repeat": repeat_number,
            "fold": fold_number,
            "task": "ordinal_1_to_5",
            "model": "Ordinal Logistic Regression",
            "feature_group": group,
            **evaluate_ordinal_predictions(cv_y_valid_ord, ordinal_prediction),
        })

        binary_specs = [
            ("Binary Logistic Regression", LogisticRegression(max_iter=3000, solver="lbfgs"), scaled_train, scaled_valid),
            ("Naive Bayes", GaussianNB(), scaled_train, scaled_valid),
            ("Decision Tree", DecisionTreeClassifier(
                max_depth=4, min_samples_leaf=300, random_state=RANDOM_STATE
            ), tree_train, tree_valid),
            ("Random Forest", RandomForestClassifier(
                n_estimators=200,
                min_samples_leaf=50,
                class_weight="balanced_subsample",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ), tree_train, tree_valid),
            ("XGBoost", XGBClassifier(
                n_estimators=250,
                max_depth=3,
                learning_rate=0.05,
                subsample=0.9,
                colsample_bytree=0.9,
                objective="binary:logistic",
                eval_metric="logloss",
                random_state=RANDOM_STATE,
                n_jobs=-1,
                tree_method="hist",
            ), tree_train, tree_valid),
        ]

        for model_name, classifier, fit_matrix, validation_matrix in binary_specs:
            classifier.fit(fit_matrix, cv_y_train_bin)
            binary_prediction = classifier.predict(validation_matrix)
            binary_probability = classifier.predict_proba(validation_matrix)[:, 1]
            cv_rows.append({
                "repeat": repeat_number,
                "fold": fold_number,
                "task": "binary_satisfied",
                "model": model_name,
                "feature_group": group,
                **evaluate_binary_predictions(
                    cv_y_valid_bin, binary_prediction, binary_probability
                ),
                "brier_score": brier_score_loss(cv_y_valid_bin, binary_probability),
            })

repeated_cv_fold_metrics = pd.DataFrame(cv_rows)
summary_rows = []
metrics_by_task = {
    "ordinal_1_to_5": [
        "accuracy", "macro_f1", "quadratic_weighted_kappa", "mean_absolute_error", "within_1"
    ],
    "binary_satisfied": [
        "accuracy", "balanced_accuracy", "macro_f1", "roc_auc", "brier_score"
    ],
}

for keys, model_rows in repeated_cv_fold_metrics.groupby(
    ["task", "model", "feature_group"], dropna=False
):
    summary = {
        "task": keys[0],
        "model": keys[1],
        "feature_group": keys[2],
        "n_splits": len(model_rows),
    }
    for metric in metrics_by_task[keys[0]]:
        summary[f"{metric}_mean"] = model_rows[metric].mean()
        summary[f"{metric}_sd"] = model_rows[metric].std(ddof=1)
    summary_rows.append(summary)

repeated_cv_summary = pd.DataFrame(summary_rows).sort_values(
    ["task", "model", "feature_group"]
).reset_index(drop=True)
repeated_cv_fold_metrics.to_csv(
    OUTPUT_DIR / "repeated_cv_fold_metrics_v0.1.csv", index=False
)
repeated_cv_summary.to_csv(
    OUTPUT_DIR / "repeated_cv_summary_v0.1.csv", index=False
)
repeated_cv_dimension_coefficients = pd.DataFrame(cv_dimension_coefficient_rows)
repeated_cv_dimension_coefficients.to_csv(
    OUTPUT_DIR / "repeated_cv_dimension_coefficients_v0.1.csv", index=False
)
repeated_cv_dimension_rank_summary = (
    repeated_cv_dimension_coefficients
    .groupby(["feature", "nss_module"], as_index=False)
    .agg(
        median_coefficient=("coefficient", "median"),
        minimum_coefficient=("coefficient", "min"),
        maximum_coefficient=("coefficient", "max"),
        median_rank=("dimension_rank", "median"),
        best_rank=("dimension_rank", "min"),
        worst_rank=("dimension_rank", "max"),
        first_place_folds=("dimension_rank", lambda value: int((value == 1).sum())),
        top_two_folds=("dimension_rank", lambda value: int((value <= 2).sum())),
    )
    .sort_values(["median_rank", "nss_module"])
    .reset_index(drop=True)
)
repeated_cv_dimension_rank_summary.to_csv(
    OUTPUT_DIR / "repeated_cv_dimension_rank_summary_v0.1.csv", index=False
)

display(repeated_cv_summary[repeated_cv_summary["task"] == "ordinal_1_to_5"].round(3))
display(repeated_cv_summary[repeated_cv_summary["task"] == "binary_satisfied"].round(3))
display(repeated_cv_dimension_rank_summary.round(3))

,task,model,feature_group,n_splits,accuracy_mean,accuracy_sd,balanced_accuracy_mean,balanced_accuracy_sd,macro_f1_mean,macro_f1_sd,roc_auc_mean,roc_auc_sd,brier_score_mean,brier_score_sd,quadratic_weighted_kappa_mean,quadratic_weighted_kappa_sd,mean_absolute_error_mean,mean_absolute_error_sd,within_1_mean,within_1_sd
11,ordinal_1_to_5,Baseline majority class,none,25,0.441,0.000,NaN,NaN,0.122,0.000,NaN,NaN,NaN,NaN,0.000,0.000,0.739,0.000,0.869,0.000
12,ordinal_1_to_5,Ordinal Logistic Regression,module_plus_demographic,25,0.666,0.005,NaN,NaN,0.568,0.007,NaN,NaN,NaN,NaN,0.792,0.005,0.373,0.006,0.964,0.002
13,ordinal_1_to_5,Ordinal Logistic Regression,module_scores,25,0.666,0.005,NaN,NaN,0.566,0.007,NaN,NaN,NaN,NaN,0.791,0.005,0.374,0.006,0.964,0.002


,task,model,feature_group,n_splits,accuracy_mean,accuracy_sd,balanced_accuracy_mean,balanced_accuracy_sd,macro_f1_mean,macro_f1_sd,roc_auc_mean,roc_auc_sd,brier_score_mean,brier_score_sd,quadratic_weighted_kappa_mean,quadratic_weighted_kappa_sd,mean_absolute_error_mean,mean_absolute_error_sd,within_1_mean,within_1_sd
0,binary_satisfied,Baseline majority class,none,25,0.750,0.000,0.500,0.000,0.429,0.000,0.500,0.000,0.187,0.000,NaN,NaN,NaN,NaN,NaN,NaN
1,binary_satisfied,Binary Logistic Regression,module_plus_demographic,25,0.883,0.003,0.819,0.005,0.836,0.004,0.934,0.002,0.083,0.002,NaN,NaN,NaN,NaN,NaN,NaN
2,binary_satisfied,Binary Logistic Regression,module_scores,25,0.884,0.003,0.819,0.004,0.836,0.004,0.933,0.002,0.084,0.002,NaN,NaN,NaN,NaN,NaN,NaN
3,binary_satisfied,Decision Tree,module_plus_demographic,25,0.869,0.003,0.807,0.010,0.818,0.006,0.912,0.003,0.094,0.002,NaN,NaN,NaN,NaN,NaN,NaN
4,binary_satisfied,Decision Tree,module_scores,25,0.869,0.003,0.807,0.010,0.818,0.006,0.912,0.003,0.094,0.002,NaN,NaN,NaN,NaN,NaN,NaN
5,binary_satisfied,Naive Bayes,module_plus_demographic,25,0.852,0.004,0.828,0.005,0.812,0.005,0.909,0.003,0.128,0.004,NaN,NaN,NaN,NaN,NaN,NaN
6,binary_satisfied,Naive Bayes,module_scores,25,0.868,0.003,0.840,0.004,0.829,0.004,0.928,0.002,0.112,0.003,NaN,NaN,NaN,NaN,NaN,NaN
7,binary_satisfied,Random Forest,module_plus_demographic,25,0.850,0.003,0.855,0.003,0.818,0.003,0.933,0.002,0.106,0.001,NaN,NaN,NaN,NaN,NaN,NaN
8,binary_satisfied,Random Forest,module_scores,25,0.852,0.002,0.855,0.003,0.820,0.003,0.934,0.002,0.103,0.001,NaN,NaN,NaN,NaN,NaN,NaN
9,binary_satisfied,XGBoost,module_plus_demographic,25,0.884,0.002,0.823,0.004,0.838,0.004,0.935,0.002,0.083,0.002,NaN,NaN,NaN,NaN,NaN,NaN


,feature,nss_module,median_coefficient,minimum_coefficient,maximum_coefficient,median_rank,best_rank,worst_rank,first_place_folds,top_two_folds
0,module_teaching_on_my_course,Teaching on my course,1.297,1.268,1.316,1.0,1,1,25,25
1,module_organisation_and_management,Organisation and management,0.621,0.603,0.641,2.0,2,2,0,25
2,module_learning_opportunities,Learning opportunities,0.383,0.360,0.401,3.0,3,4,0,0
3,module_student_voice,Student voice,0.379,0.362,0.394,4.0,3,4,0,0
4,module_academic_support,Academic support,0.294,0.282,0.308,5.0,5,5,0,0
5,module_learning_resources,Learning resources,0.251,0.240,0.263,6.0,6,7,0,0
6,module_learning_community,Learning community,0.251,0.229,0.265,7.0,6,7,0,0
7,module_assessment_and_feedback,Assessment and feedback,0.210,0.191,0.223,8.0,8,8,0,0


## 19. Binary probability calibration

Brier score, log loss, expected calibration error, calibration intercept and calibration slope assess whether predicted probabilities correspond to observed satisfaction rates. Logistic regression and XGBoost are evaluated on the held-out test set, and XGBoost is additionally calibrated by five-fold sigmoid scaling on the training data.

In [25]:
def calibration_summary(y_true, probability, model_name):
    probability = np.clip(np.asarray(probability, dtype=float), 1e-6, 1 - 1e-6)
    y_array = np.asarray(y_true, dtype=int)
    probability_logit = np.log(probability / (1 - probability))
    calibration_fit = sm.GLM(
        y_array,
        sm.add_constant(probability_logit),
        family=sm.families.Binomial(),
    ).fit()

    calibration_frame = pd.DataFrame({"observed": y_array, "probability": probability})
    calibration_frame["bin"] = pd.qcut(
        calibration_frame["probability"],
        q=CALIBRATION_BINS,
        labels=False,
    )
    curve = (
        calibration_frame.groupby("bin", as_index=False)
        .agg(
            n=("observed", "size"),
            mean_predicted_probability=("probability", "mean"),
            observed_satisfaction_rate=("observed", "mean"),
        )
    )
    curve["absolute_gap"] = (
        curve["mean_predicted_probability"] - curve["observed_satisfaction_rate"]
    ).abs()
    expected_calibration_error = np.average(curve["absolute_gap"], weights=curve["n"])
    curve.insert(0, "model", model_name)

    metrics = {
        "model": model_name,
        "roc_auc": roc_auc_score(y_array, probability),
        "brier_score": brier_score_loss(y_array, probability),
        "log_loss": log_loss(y_array, probability),
        "expected_calibration_error": expected_calibration_error,
        "calibration_intercept": calibration_fit.params[0],
        "calibration_slope": calibration_fit.params[1],
    }
    return metrics, curve


scaled_train, scaled_test, _, _ = scaled_design_matrices["module_plus_demographic"]
tree_train, tree_test, _, _ = tree_design_matrices["module_plus_demographic"]
logistic_probability = binary_logistic_artifacts["module_plus_demographic"]["classifier"].predict_proba(
    scaled_test
)[:, 1]
xgboost_probability = binary_model_artifacts[("XGBoost", "module_plus_demographic")]["classifier"].predict_proba(
    tree_test
)[:, 1]

calibrated_xgboost_pipeline = Pipeline([
    ("preprocessor", make_preprocessor("module_plus_demographic", scale_numeric=False)),
    ("classifier", XGBClassifier(
        n_estimators=250,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=1,
        tree_method="hist",
    )),
])
calibrated_xgboost = CalibratedClassifierCV(
    estimator=calibrated_xgboost_pipeline,
    method="sigmoid",
    cv=5,
    n_jobs=1,
)
calibrated_xgboost.fit(train_df, y_train_bin)
calibrated_xgboost_probability = calibrated_xgboost.predict_proba(test_df)[:, 1]

calibration_metrics = []
calibration_curves = []
for model_name, probability in [
    ("Binary Logistic Regression", logistic_probability),
    ("XGBoost", xgboost_probability),
    ("XGBoost with sigmoid calibration", calibrated_xgboost_probability),
]:
    metrics, curve = calibration_summary(y_test_bin, probability, model_name)
    calibration_metrics.append(metrics)
    calibration_curves.append(curve)

binary_calibration_metrics = pd.DataFrame(calibration_metrics).sort_values("brier_score")
binary_calibration_curve = pd.concat(calibration_curves, ignore_index=True)
binary_calibration_metrics.to_csv(
    OUTPUT_DIR / "binary_calibration_metrics_v0.1.csv", index=False
)
binary_calibration_curve.to_csv(
    OUTPUT_DIR / "binary_calibration_curve_v0.1.csv", index=False
)

display(binary_calibration_metrics.round(4))
display(binary_calibration_curve.round(4))

,model,roc_auc,brier_score,log_loss,expected_calibration_error,calibration_intercept,calibration_slope
1,XGBoost,0.9396,0.0806,0.2623,0.0062,-0.0511,1.0384
0,Binary Logistic Regression,0.9386,0.0810,0.2647,0.0115,-0.0522,1.0398
2,XGBoost with sigmoid calibration,0.9398,0.0826,0.2773,0.0364,-0.0715,1.0415


,model,bin,n,mean_predicted_probability,observed_satisfaction_rate,absolute_gap
0,Binary Logistic Regression,0,1013,0.0369,0.0503,0.0135
1,Binary Logistic Regression,1,1013,0.2936,0.2557,0.0379
2,Binary Logistic Regression,2,1012,0.6145,0.5978,0.0166
3,Binary Logistic Regression,3,1013,0.8005,0.7887,0.0118
4,Binary Logistic Regression,4,1013,0.8923,0.9013,0.0090
5,Binary Logistic Regression,5,1012,0.9385,0.9496,0.0111
6,Binary Logistic Regression,6,1013,0.9686,0.9793,0.0107
7,Binary Logistic Regression,7,1012,0.9845,0.9862,0.0016
8,Binary Logistic Regression,8,1013,0.9933,0.9951,0.0018
9,Binary Logistic Regression,9,1013,0.9979,0.9990,0.0011


## 20. Validation interpretation

Internal consistency supports or qualifies the construction of the eight mean scores. The proportional-odds diagnostic determines whether one common ordinal slope is adequate across thresholds. Repeated cross-validation quantifies split-to-split variation, while calibration separates probability accuracy from ranking performance. All results remain associational because Q1-Q27 are collected contemporaneously in the same survey.

In [26]:
global_po = proportional_odds_diagnostic.loc[
    proportional_odds_diagnostic["feature"] == "__GLOBAL__"
].iloc[0]
best_binary_cv = (
    repeated_cv_summary[
        (repeated_cv_summary["task"] == "binary_satisfied")
        & (repeated_cv_summary["feature_group"] == "module_plus_demographic")
    ]
    .sort_values("roc_auc_mean", ascending=False)
    .iloc[0]
)
best_calibration = binary_calibration_metrics.sort_values("brier_score").iloc[0]

print("Internal consistency across the eight dimensions:")
display(module_reliability[[
    "nss_module",
    "n_items",
    "cronbach_alpha",
    "standardised_alpha",
    "spearman_brown_two_item",
]].round(3))

print(
    f"Global proportional-odds Wald test: chi-square={global_po['wald_chi_square']:.1f}, "
    f"df={int(global_po['degrees_of_freedom'])}, p={global_po['p_value']:.3g}."
)
print(
    f"Best repeated-CV binary ROC-AUC: {best_binary_cv['model']} "
    f"({best_binary_cv['feature_group']}), mean={best_binary_cv['roc_auc_mean']:.3f}, "
    f"SD={best_binary_cv['roc_auc_sd']:.3f}."
)
print(
    f"Lowest held-out Brier score: {best_calibration['model']}, "
    f"Brier={best_calibration['brier_score']:.4f}, "
    f"slope={best_calibration['calibration_slope']:.3f}."
)
print(
    "Interpretation rule: validation checks quantify reliability, assumptions and predictive stability; "
    "they do not convert survey associations into causal effects."
)

Internal consistency across the eight dimensions:


,nss_module,n_items,cronbach_alpha,standardised_alpha,spearman_brown_two_item
0,Teaching on my course,4,0.837,0.839,NaN
1,Learning opportunities,3,0.781,0.781,NaN
2,Assessment and feedback,4,0.815,0.816,NaN
3,Academic support,3,0.815,0.814,NaN
4,Organisation and management,3,0.739,0.738,NaN
5,Learning resources,3,0.786,0.786,NaN
6,Learning community,2,0.685,0.698,0.698
7,Student voice,4,0.810,0.810,NaN


Global proportional-odds Wald test: chi-square=312.9, df=24, p=4.23e-52.
Best repeated-CV binary ROC-AUC: XGBoost (module_plus_demographic), mean=0.935, SD=0.002.
Lowest held-out Brier score: XGBoost, Brier=0.0806, slope=1.038.
Interpretation rule: validation checks quantify reliability, assumptions and predictive stability; they do not convert survey associations into causal effects.


## 21. Exact-duplicate sensitivity analysis

Exact duplicate rows may place identical response profiles in both the training and test sets. This sensitivity analysis removes exact duplicates before outcome filtering and data splitting, then repeats the central ordinal and binary comparisons with the same preprocessing, random seed and model settings. Stability is assessed through changes in holdout performance and in the coefficients, 95% confidence intervals and ranks of the eight factors in the primary ordinal model. Because the deduplicated data are re-stratified after rows are removed, performance changes reflect both duplicate removal and re-partitioning; they are not an isolated estimate of duplicate leakage.

In [27]:
analytic_raw = df_raw.loc[df_raw[target_col] != 0].copy()
analytic_hash = pd.Series(
    pd.util.hash_pandas_object(analytic_raw, index=False).to_numpy(),
    index=analytic_raw.index,
)
train_hashes = set(analytic_hash.loc[train_idx].tolist())
test_rows_matching_train = int(analytic_hash.loc[test_idx].isin(train_hashes).sum())

deduplicated_raw = df_raw.drop_duplicates(keep="first").copy()
deduplicated_modules = clean_demographics(add_module_scores(deduplicated_raw))
deduplicated_modules = deduplicated_modules.loc[
    deduplicated_modules[target_col] != 0
].copy()
deduplicated_y_ordinal = deduplicated_modules[target_col].astype(int)
deduplicated_y_binary = deduplicated_y_ordinal.isin([4, 5]).astype(int)

dedup_train_idx, dedup_test_idx = train_test_split(
    deduplicated_modules.index,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=deduplicated_y_ordinal,
)
dedup_train = deduplicated_modules.loc[dedup_train_idx]
dedup_test = deduplicated_modules.loc[dedup_test_idx]
dedup_y_train_ord = deduplicated_y_ordinal.loc[dedup_train_idx]
dedup_y_test_ord = deduplicated_y_ordinal.loc[dedup_test_idx]
dedup_y_train_bin = deduplicated_y_binary.loc[dedup_train_idx]
dedup_y_test_bin = deduplicated_y_binary.loc[dedup_test_idx]

duplicate_sample_summary = pd.DataFrame([{
    "raw_rows": len(df_raw),
    "raw_exact_duplicates_removed": int(df_raw.duplicated(keep="first").sum()),
    "analytic_rows_original": len(df_modules),
    "analytic_exact_duplicates_removed": int(analytic_raw.duplicated(keep="first").sum()),
    "analytic_rows_deduplicated": len(deduplicated_modules),
    "original_test_rows_matching_a_training_signature": test_rows_matching_train,
    "original_test_match_rate": test_rows_matching_train / len(test_idx),
}])
duplicate_sample_summary.to_csv(
    OUTPUT_DIR / "duplicate_sensitivity_sample_v0.1.csv", index=False
)

dedup_scaled = {
    group: build_design_matrices(dedup_train, dedup_test, group, scale_numeric=True)
    for group in ["module_scores", "module_plus_demographic"]
}
dedup_tree = {
    group: build_design_matrices(dedup_train, dedup_test, group, scale_numeric=False)
    for group in ["module_scores", "module_plus_demographic"]
}

dedup_metric_lookup = {}
dedup_ordinal_artifacts = {}
for group, (X_train, X_test, feature_names, preprocessor) in dedup_scaled.items():
    model = OrderedModel(dedup_y_train_ord, X_train, distr="logit")
    result = model.fit(method="lbfgs", maxiter=ORDINAL_MAXITER, disp=False)
    probability = result.model.predict(result.params, exog=X_test)
    prediction = np.asarray(probability).argmax(axis=1) + 1
    dedup_metric_lookup[("ordinal_1_to_5", "Ordinal Logistic Regression", group)] = (
        evaluate_ordinal_predictions(dedup_y_test_ord, prediction)
    )
    dedup_ordinal_artifacts[group] = {
        "result": result,
        "feature_names": feature_names,
        "preprocessor": preprocessor,
    }

for group in ["module_scores", "module_plus_demographic"]:
    scaled_train, scaled_test, _, _ = dedup_scaled[group]
    logistic = LogisticRegression(max_iter=3000, solver="lbfgs")
    logistic.fit(scaled_train, dedup_y_train_bin)
    logistic_prediction = logistic.predict(scaled_test)
    logistic_probability = logistic.predict_proba(scaled_test)[:, 1]
    dedup_metric_lookup[("binary_satisfied", "Binary Logistic Regression", group)] = (
        evaluate_binary_predictions(
            dedup_y_test_bin, logistic_prediction, logistic_probability
        )
    )

    tree_train, tree_test, _, _ = dedup_tree[group]
    xgboost = XGBClassifier(
        n_estimators=250,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        tree_method="hist",
    )
    xgboost.fit(tree_train, dedup_y_train_bin)
    xgboost_prediction = xgboost.predict(tree_test)
    xgboost_probability = xgboost.predict_proba(tree_test)[:, 1]
    dedup_metric_lookup[("binary_satisfied", "XGBoost", group)] = (
        evaluate_binary_predictions(
            dedup_y_test_bin, xgboost_prediction, xgboost_probability
        )
    )

comparison_specs = [
    ("ordinal_1_to_5", "Ordinal Logistic Regression", "module_scores"),
    ("ordinal_1_to_5", "Ordinal Logistic Regression", "module_plus_demographic"),
    ("binary_satisfied", "Binary Logistic Regression", "module_scores"),
    ("binary_satisfied", "Binary Logistic Regression", "module_plus_demographic"),
    ("binary_satisfied", "XGBoost", "module_scores"),
    ("binary_satisfied", "XGBoost", "module_plus_demographic"),
]
comparison_metrics = [
    "accuracy", "balanced_accuracy", "macro_f1", "quadratic_weighted_kappa",
    "mean_absolute_error", "roc_auc",
]
performance_rows = []
for task, model_name, group in comparison_specs:
    original = combined_results.loc[
        (combined_results["task"] == task)
        & (combined_results["model"] == model_name)
        & (combined_results["feature_group"] == group)
    ].iloc[0]
    deduplicated = dedup_metric_lookup[(task, model_name, group)]
    row = {
        "task": task,
        "model": model_name,
        "feature_group": group,
        "original_n": len(df_modules),
        "deduplicated_n": len(deduplicated_modules),
    }
    for metric in comparison_metrics:
        original_value = original.get(metric, np.nan)
        deduplicated_value = deduplicated.get(metric, np.nan)
        row[f"original_{metric}"] = original_value
        row[f"deduplicated_{metric}"] = deduplicated_value
        row[f"change_{metric}"] = deduplicated_value - original_value
    performance_rows.append(row)

duplicate_performance = pd.DataFrame(performance_rows)
duplicate_performance.to_csv(
    OUTPUT_DIR / "duplicate_sensitivity_performance_v0.1.csv", index=False
)

original_primary = ordinal_artifacts["module_scores"]
deduplicated_primary = dedup_ordinal_artifacts["module_scores"]
feature_names = original_primary["feature_names"]
n_features = len(feature_names)

original_coefficients = np.asarray(
    original_primary["result"].params[:n_features], dtype=float
)
original_standard_errors = np.asarray(
    original_primary["result"].bse[:n_features], dtype=float
)
deduplicated_coefficients = np.asarray(
    deduplicated_primary["result"].params[:n_features], dtype=float
)
deduplicated_standard_errors = np.asarray(
    deduplicated_primary["result"].bse[:n_features], dtype=float
)

duplicate_coefficients = pd.DataFrame({
    "feature": feature_names,
    "original_coefficient": original_coefficients,
    "original_ci_lower": original_coefficients - 1.96 * original_standard_errors,
    "original_ci_upper": original_coefficients + 1.96 * original_standard_errors,
    "deduplicated_coefficient": deduplicated_coefficients,
    "deduplicated_ci_lower": deduplicated_coefficients - 1.96 * deduplicated_standard_errors,
    "deduplicated_ci_upper": deduplicated_coefficients + 1.96 * deduplicated_standard_errors,
})
duplicate_coefficients["nss_module"] = duplicate_coefficients["feature"].map(
    module_name_by_col
)
duplicate_coefficients["question_range"] = duplicate_coefficients["feature"].map(
    module_range_by_col
)
duplicate_coefficients["coefficient_change"] = (
    duplicate_coefficients["deduplicated_coefficient"]
    - duplicate_coefficients["original_coefficient"]
)
duplicate_coefficients["original_rank"] = (
    duplicate_coefficients["original_coefficient"].abs()
    .rank(ascending=False, method="min").astype(int)
)
duplicate_coefficients["deduplicated_rank"] = (
    duplicate_coefficients["deduplicated_coefficient"].abs()
    .rank(ascending=False, method="min").astype(int)
)
duplicate_coefficients["rank_change"] = (
    duplicate_coefficients["deduplicated_rank"]
    - duplicate_coefficients["original_rank"]
)
duplicate_coefficients["original_odds_ratio"] = np.exp(
    duplicate_coefficients["original_coefficient"]
)
duplicate_coefficients["deduplicated_odds_ratio"] = np.exp(
    duplicate_coefficients["deduplicated_coefficient"]
)
duplicate_coefficients = duplicate_coefficients.sort_values(
    "deduplicated_rank"
).reset_index(drop=True)
duplicate_coefficients.to_csv(
    OUTPUT_DIR / "duplicate_sensitivity_ordinal_coefficients_v0.1.csv", index=False
)

ordinal_factor_row = duplicate_performance.loc[
    (duplicate_performance["task"] == "ordinal_1_to_5")
    & (duplicate_performance["feature_group"] == "module_scores")
].iloc[0]
binary_xgboost_row = duplicate_performance.loc[
    (duplicate_performance["task"] == "binary_satisfied")
    & (duplicate_performance["model"] == "XGBoost")
    & (duplicate_performance["feature_group"] == "module_plus_demographic")
].iloc[0]
rank_correlation = duplicate_coefficients[
    ["original_rank", "deduplicated_rank"]
].corr(method="spearman").iloc[0, 1]

display(duplicate_sample_summary)
display(duplicate_performance.round(4))
display(duplicate_coefficients.round(4))
print(
    f"After exact duplicates are removed, primary ordinal QWK changes from "
    f"{ordinal_factor_row['original_quadratic_weighted_kappa']:.3f} to "
    f"{ordinal_factor_row['deduplicated_quadratic_weighted_kappa']:.3f}."
)
print(
    f"Combined XGBoost ROC-AUC changes from "
    f"{binary_xgboost_row['original_roc_auc']:.3f} to "
    f"{binary_xgboost_row['deduplicated_roc_auc']:.3f}."
)
print(
    f"The Spearman correlation between original and deduplicated factor ranks is "
    f"{rank_correlation:.3f}."
)


,raw_rows,raw_exact_duplicates_removed,analytic_rows_original,analytic_exact_duplicates_removed,analytic_rows_deduplicated,original_test_rows_matching_a_training_signature,original_test_match_rate
0,50915,1454,50633,1454,49179,341,0.033672


,task,model,feature_group,original_n,deduplicated_n,original_accuracy,deduplicated_accuracy,change_accuracy,original_balanced_accuracy,deduplicated_balanced_accuracy,...,change_macro_f1,original_quadratic_weighted_kappa,deduplicated_quadratic_weighted_kappa,change_quadratic_weighted_kappa,original_mean_absolute_error,deduplicated_mean_absolute_error,change_mean_absolute_error,original_roc_auc,deduplicated_roc_auc,change_roc_auc
0,ordinal_1_to_5,Ordinal Logistic Regression,module_scores,50633,49179,0.6638,0.6582,-0.0056,NaN,NaN,...,-0.0032,0.7921,0.7889,-0.0032,0.3758,0.3817,0.0058,NaN,NaN,NaN
1,ordinal_1_to_5,Ordinal Logistic Regression,module_plus_demographic,50633,49179,0.6627,0.6594,-0.0033,NaN,NaN,...,-0.0026,0.7914,0.7903,-0.0012,0.3771,0.3800,0.0029,NaN,NaN,NaN
2,binary_satisfied,Binary Logistic Regression,module_scores,50633,49179,0.8896,0.8824,-0.0072,0.8251,0.8229,...,-0.0058,NaN,NaN,NaN,NaN,NaN,NaN,0.9384,0.9317,-0.0067
3,binary_satisfied,Binary Logistic Regression,module_plus_demographic,50633,49179,0.8906,0.8840,-0.0066,0.8258,0.8252,...,-0.0046,NaN,NaN,NaN,NaN,NaN,NaN,0.9386,0.9322,-0.0064
4,binary_satisfied,XGBoost,module_scores,50633,49179,0.8876,0.8805,-0.0071,0.8268,0.8245,...,-0.0056,NaN,NaN,NaN,NaN,NaN,NaN,0.9392,0.9328,-0.0063
5,binary_satisfied,XGBoost,module_plus_demographic,50633,49179,0.8891,0.8822,-0.0069,0.8285,0.8270,...,-0.0052,NaN,NaN,NaN,NaN,NaN,NaN,0.9396,0.9335,-0.0060


,feature,original_coefficient,original_ci_lower,original_ci_upper,deduplicated_coefficient,deduplicated_ci_lower,deduplicated_ci_upper,nss_module,question_range,coefficient_change,original_rank,deduplicated_rank,rank_change,original_odds_ratio,deduplicated_odds_ratio
0,module_teaching_on_my_course,1.2944,1.2572,1.3315,1.2985,1.2618,1.3352,Teaching on my course,Q1-Q4,0.0042,1,1,0,3.6486,3.6638
1,module_organisation_and_management,0.6199,0.5878,0.6520,0.6189,0.5871,0.6507,Organisation and management,Q15-Q17,-0.0009,2,2,0,1.8587,1.8569
2,module_student_voice,0.3776,0.3434,0.4118,0.3712,0.3375,0.4048,Student voice,Q23-Q26,-0.0064,4,3,-1,1.4588,1.4494
3,module_learning_opportunities,0.3836,0.3512,0.4161,0.3399,0.3078,0.3720,Learning opportunities,Q5-Q7,-0.0437,3,4,1,1.4676,1.4048
4,module_academic_support,0.2940,0.2616,0.3264,0.2702,0.2382,0.3022,Academic support,Q12-Q14,-0.0238,5,5,0,1.3418,1.3103
5,module_learning_community,0.2496,0.2203,0.2790,0.2595,0.2305,0.2886,Learning community,Q21-Q22,0.0099,7,6,-1,1.2836,1.2963
6,module_learning_resources,0.2501,0.2238,0.2765,0.2367,0.2107,0.2628,Learning resources,Q18-Q20,-0.0134,6,7,1,1.2842,1.2671
7,module_assessment_and_feedback,0.2094,0.1790,0.2398,0.1958,0.1659,0.2256,Assessment and feedback,Q8-Q11,-0.0137,8,8,0,1.2330,1.2162


After exact duplicates are removed, primary ordinal QWK changes from 0.792 to 0.789.
Combined XGBoost ROC-AUC changes from 0.940 to 0.934.
The Spearman correlation between original and deduplicated factor ranks is 0.952.


### 21.1 Sensitivity result

Removing 1,454 duplicated analytic observations beyond the first occurrence reduced the sample from 50,633 to 49,179. In the original split, 341 test records (3.37%) had an exact response signature present in the training set. After duplicate observations were removed before splitting, QWK for the primary dimension-only ordinal model changed from 0.792 to 0.789. ROC-AUC for combined XGBoost changed from 0.940 to 0.934. The Spearman correlation between the original and deduplicated ordinal coefficient ranks was 0.952. Teaching on my course and Organisation and management remained first and second, and every deduplicated ordinal coefficient remained positive with a 95% confidence interval above zero. Duplicate overlap slightly increased apparent performance without changing the principal substantive conclusion.

## 22. Temporal transport validation

Models are trained on the 2018-2021 surveys and evaluated on the year-held-out 2022 survey. This evaluates transport to a later survey year rather than forecasting future student outcomes: Q1-Q26 and Q27 are still measured in the same response occasion. Survey year is excluded from the combined specification because 2022 is absent from training and an unseen one-hot category would otherwise map implicitly to the reference year. The dimension-only ordinal coefficients and ranks from the pre-2022 fit are exported separately; they describe the model learned from 2018-2021, while the performance metrics assess its transport to 2022.


In [28]:
def build_custom_design_matrices(
    train_frame, test_frame, categorical_columns, numeric_columns, scale_numeric=True
):
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler() if scale_numeric else "passthrough"),
    ])
    transformers = [
        ("categorical", OneHotEncoder(
            drop="first", handle_unknown="ignore", sparse_output=False
        ), categorical_columns),
        ("numeric", numeric_pipeline, numeric_columns),
    ]
    preprocessor = ColumnTransformer(
        [transformer for transformer in transformers if transformer[2]],
        remainder="drop",
        verbose_feature_names_out=False,
    )
    X_train = np.asarray(preprocessor.fit_transform(train_frame), dtype=float)
    X_test = np.asarray(preprocessor.transform(test_frame), dtype=float)
    return X_train, X_test, preprocessor.get_feature_names_out().tolist(), preprocessor


def make_binary_model_specs(model_names=None):
    specs = [
        ("Binary Logistic Regression", LogisticRegression(max_iter=3000, solver="lbfgs"), "scaled"),
        ("Naive Bayes", GaussianNB(), "scaled"),
        ("Decision Tree", DecisionTreeClassifier(
            max_depth=4, min_samples_leaf=300, random_state=RANDOM_STATE
        ), "tree"),
        ("Random Forest", RandomForestClassifier(
            n_estimators=200, min_samples_leaf=50,
            class_weight="balanced_subsample", random_state=RANDOM_STATE, n_jobs=-1
        ), "tree"),
        ("XGBoost", XGBClassifier(
            n_estimators=250, max_depth=3, learning_rate=0.05,
            subsample=0.9, colsample_bytree=0.9,
            objective="binary:logistic", eval_metric="logloss",
            random_state=RANDOM_STATE, n_jobs=-1, tree_method="hist"
        ), "tree"),
    ]
    return specs if model_names is None else [
        spec for spec in specs if spec[0] in model_names
    ]


def temporal_result(task, model, feature_group, metrics):
    return {
        "task": task,
        "model": model,
        "feature_group": feature_group,
        "train_years": "2018-2021",
        "test_year": 2022,
        "train_n": len(temporal_train),
        "test_n": len(temporal_test),
        **metrics,
    }


temporal_train = df_modules.loc[df_modules["survey_year"].astype(int) <= 2021].copy()
temporal_test = df_modules.loc[df_modules["survey_year"].astype(int) == 2022].copy()
temporal_y_train_ord = temporal_train[target_col].astype(int)
temporal_y_test_ord = temporal_test[target_col].astype(int)
temporal_y_train_bin = temporal_y_train_ord.isin([4, 5]).astype(int)
temporal_y_test_bin = temporal_y_test_ord.isin([4, 5]).astype(int)

temporal_groups = {
    "module_scores": ([], module_score_cols),
    "module_plus_demographic_no_year": (
        [column for column in demographic_cols if column != "survey_year"],
        module_score_cols,
    ),
}
temporal_scaled = {
    group: build_custom_design_matrices(
        temporal_train, temporal_test, categorical, numeric, scale_numeric=True
    )
    for group, (categorical, numeric) in temporal_groups.items()
}
temporal_tree = {
    group: build_custom_design_matrices(
        temporal_train, temporal_test, categorical, numeric, scale_numeric=False
    )
    for group, (categorical, numeric) in temporal_groups.items()
}

ordinal_majority = int(temporal_y_train_ord.mode().iloc[0])
binary_majority = int(temporal_y_train_bin.mode().iloc[0])
binary_baseline_probability = np.full(
    len(temporal_y_test_bin), temporal_y_train_bin.mean()
)
temporal_rows = []
temporal_dimension_coefficient_rows = []
temporal_rows = [
    temporal_result(
        "ordinal_1_to_5", "Baseline majority class", "none",
        evaluate_ordinal_predictions(
            temporal_y_test_ord,
            np.full(len(temporal_y_test_ord), ordinal_majority, dtype=int),
        ),
    ),
    temporal_result(
        "binary_satisfied", "Baseline majority class", "none",
        {
            **evaluate_binary_predictions(
                temporal_y_test_bin,
                np.full(len(temporal_y_test_bin), binary_majority, dtype=int),
                binary_baseline_probability,
            ),
            "brier_score": brier_score_loss(
                temporal_y_test_bin, binary_baseline_probability
            ),
        },
    ),
]

for group in temporal_groups:
    scaled_train, scaled_test, _, _ = temporal_scaled[group]
    tree_train, tree_test, _, _ = temporal_tree[group]

    ordinal_model = OrderedModel(temporal_y_train_ord, scaled_train, distr="logit")
    ordinal_result = ordinal_model.fit(
        method="lbfgs", maxiter=ORDINAL_MAXITER, disp=False
    )
    ordinal_probability = ordinal_result.model.predict(
        ordinal_result.params, exog=scaled_test
    )
    if group == "module_scores":
        temporal_coefficients = np.asarray(
            ordinal_result.params[:len(module_score_cols)], dtype=float
        )
        temporal_ranks = (
            pd.Series(np.abs(temporal_coefficients))
            .rank(ascending=False, method="min").astype(int).to_numpy()
        )
        temporal_dimension_coefficient_rows.extend([
            {
                "feature": feature,
                "nss_module": module_name_by_col[feature],
                "coefficient": coefficient,
                "odds_ratio": float(np.exp(coefficient)),
                "dimension_rank": int(rank),
                "train_years": "2018-2021",
                "test_year": 2022,
            }
            for feature, coefficient, rank in zip(
                module_score_cols, temporal_coefficients, temporal_ranks
            )
        ])
    temporal_rows.append(temporal_result(
        "ordinal_1_to_5", "Ordinal Logistic Regression", group,
        evaluate_ordinal_predictions(
            temporal_y_test_ord,
            np.asarray(ordinal_probability).argmax(axis=1) + 1,
        ),
    ))

    matrices = {"scaled": (scaled_train, scaled_test), "tree": (tree_train, tree_test)}
    for model_name, classifier, matrix_kind in make_binary_model_specs():
        fit_matrix, test_matrix = matrices[matrix_kind]
        classifier.fit(fit_matrix, temporal_y_train_bin)
        prediction = classifier.predict(test_matrix)
        probability = classifier.predict_proba(test_matrix)[:, 1]
        temporal_rows.append(temporal_result(
            "binary_satisfied", model_name, group,
            {
                **evaluate_binary_predictions(
                    temporal_y_test_bin, prediction, probability
                ),
                "brier_score": brier_score_loss(
                    temporal_y_test_bin, probability
                ),
            },
        ))

temporal_validation_results = tidy_metric_rows(temporal_rows)
temporal_sample_summary = pd.DataFrame([
    {
        "period": "training", "years": "2018-2021", "n": len(temporal_train),
        "satisfied_rate": temporal_y_train_bin.mean(),
        "mean_q27": temporal_y_train_ord.mean(),
    },
    {
        "period": "test", "years": "2022", "n": len(temporal_test),
        "satisfied_rate": temporal_y_test_bin.mean(),
        "mean_q27": temporal_y_test_ord.mean(),
    },
])
temporal_validation_results.to_csv(
    OUTPUT_DIR / "temporal_validation_results_v0.1.csv", index=False
)
temporal_sample_summary.to_csv(
    OUTPUT_DIR / "temporal_validation_sample_v0.1.csv", index=False
)
temporal_dimension_coefficients = pd.DataFrame(temporal_dimension_coefficient_rows)
temporal_dimension_coefficients.to_csv(
    OUTPUT_DIR / "temporal_dimension_coefficients_v0.1.csv", index=False
)

display(temporal_sample_summary.round(4))
display(temporal_validation_results.loc[
    temporal_validation_results["task"] == "ordinal_1_to_5",
    ["model", "feature_group", "accuracy", "macro_f1", "quadratic_weighted_kappa", "mean_absolute_error"],
].round(4))
display(temporal_validation_results.loc[
    temporal_validation_results["task"] == "binary_satisfied",
    ["model", "feature_group", "accuracy", "balanced_accuracy", "macro_f1", "roc_auc", "brier_score"],
].round(4))
display(temporal_dimension_coefficients.sort_values("dimension_rank").round(4))

,period,years,n,satisfied_rate,mean_q27
0,training,2018-2021,39279,0.7599,3.9059
1,test,2022,11354,0.7167,3.7875


,model,feature_group,accuracy,macro_f1,quadratic_weighted_kappa,mean_absolute_error
11,Baseline majority class,none,0.4356,0.1214,0.0000,0.7746
12,Ordinal Logistic Regression,module_plus_demographic_no_year,0.6559,0.5705,0.7949,0.3885
13,Ordinal Logistic Regression,module_scores,0.6543,0.5688,0.7937,0.3904


,model,feature_group,accuracy,balanced_accuracy,macro_f1,roc_auc,brier_score
0,Baseline majority class,none,0.7167,0.5000,0.4175,0.5000,0.2049
1,Binary Logistic Regression,module_plus_demographic_no_year,0.8750,0.8201,0.8372,0.9333,0.0892
2,Binary Logistic Regression,module_scores,0.8751,0.8205,0.8374,0.9332,0.0892
3,Decision Tree,module_plus_demographic_no_year,0.8556,0.7789,0.8036,0.9140,0.0999
4,Decision Tree,module_scores,0.8556,0.7789,0.8036,0.9140,0.0999
5,Naive Bayes,module_plus_demographic_no_year,0.8419,0.8267,0.8130,0.9095,0.1366
6,Naive Bayes,module_scores,0.8599,0.8415,0.8321,0.9272,0.1194
7,Random Forest,module_plus_demographic_no_year,0.8428,0.8531,0.8214,0.9330,0.1096
8,Random Forest,module_scores,0.8459,0.8536,0.8241,0.9337,0.1065
9,XGBoost,module_plus_demographic_no_year,0.8754,0.8234,0.8387,0.9353,0.0882


,feature,nss_module,coefficient,odds_ratio,dimension_rank,train_years,test_year
0,module_teaching_on_my_course,Teaching on my course,1.3025,3.6785,1,2018-2021,2022
4,module_organisation_and_management,Organisation and management,0.6113,1.8428,2,2018-2021,2022
1,module_learning_opportunities,Learning opportunities,0.3722,1.4510,3,2018-2021,2022
7,module_student_voice,Student voice,0.3674,1.4440,4,2018-2021,2022
3,module_academic_support,Academic support,0.2691,1.3088,5,2018-2021,2022
6,module_learning_community,Learning community,0.2665,1.3054,6,2018-2021,2022
5,module_learning_resources,Learning resources,0.2487,1.2824,7,2018-2021,2022
2,module_assessment_and_feedback,Assessment and feedback,0.2171,1.2425,8,2018-2021,2022


## 23. Leave-one-dimension-out ablation analysis

Each NSS dimension is removed in turn while the original random train/test split is preserved. A positive performance loss means that omitting the dimension reduced out-of-sample performance. The ordinal model uses QWK; the binary logistic and XGBoost models use ROC-AUC. Ablation measures unique predictive contribution conditional on the remaining correlated dimensions, so it is complementary to coefficients and feature importance.

In [29]:
def ablation_result(task, model, omitted_feature, primary_metric, metrics, full):
    return {
        "task": task,
        "model": model,
        "omitted_feature": omitted_feature,
        "omitted_dimension": module_name_by_col[omitted_feature],
        "question_range": module_range_by_col[omitted_feature],
        "primary_metric": primary_metric,
        "full_primary_metric": full[primary_metric],
        "ablated_primary_metric": metrics[primary_metric],
        "performance_loss": full[primary_metric] - metrics[primary_metric],
        "full_accuracy": full["accuracy"],
        "ablated_accuracy": metrics["accuracy"],
        "full_macro_f1": full["macro_f1"],
        "ablated_macro_f1": metrics["macro_f1"],
    }


full_model_lookup = {
    (model, task): combined_results.loc[
        (combined_results["task"] == task)
        & (combined_results["model"] == model)
        & (combined_results["feature_group"] == "module_scores")
    ].iloc[0]
    for model, task in [
        ("Ordinal Logistic Regression", "ordinal_1_to_5"),
        ("Binary Logistic Regression", "binary_satisfied"),
        ("XGBoost", "binary_satisfied"),
    ]
}

ablation_rows = []
for omitted_feature in module_score_cols:
    retained_features = [
        feature for feature in module_score_cols if feature != omitted_feature
    ]
    scaled_train, scaled_test, _, _ = build_custom_design_matrices(
        train_df, test_df, [], retained_features, scale_numeric=True
    )
    tree_train, tree_test, _, _ = build_custom_design_matrices(
        train_df, test_df, [], retained_features, scale_numeric=False
    )

    ordinal_model = OrderedModel(y_train_ord, scaled_train, distr="logit")
    ordinal_result = ordinal_model.fit(
        method="lbfgs", maxiter=ORDINAL_MAXITER, disp=False
    )
    ordinal_probability = ordinal_result.model.predict(
        ordinal_result.params, exog=scaled_test
    )
    ordinal_metrics = evaluate_ordinal_predictions(
        y_test_ord, np.asarray(ordinal_probability).argmax(axis=1) + 1
    )
    ordinal_key = ("Ordinal Logistic Regression", "ordinal_1_to_5")
    ablation_rows.append(ablation_result(
        "ordinal_1_to_5", ordinal_key[0], omitted_feature,
        "quadratic_weighted_kappa", ordinal_metrics, full_model_lookup[ordinal_key],
    ))

    matrices = {"scaled": (scaled_train, scaled_test), "tree": (tree_train, tree_test)}
    for model_name, classifier, matrix_kind in make_binary_model_specs({
        "Binary Logistic Regression", "XGBoost"
    }):
        fit_matrix, test_matrix = matrices[matrix_kind]
        classifier.fit(fit_matrix, y_train_bin)
        prediction = classifier.predict(test_matrix)
        probability = classifier.predict_proba(test_matrix)[:, 1]
        metrics = evaluate_binary_predictions(y_test_bin, prediction, probability)
        key = (model_name, "binary_satisfied")
        ablation_rows.append(ablation_result(
            "binary_satisfied", model_name, omitted_feature,
            "roc_auc", metrics, full_model_lookup[key],
        ))

dimension_ablation_results = pd.DataFrame(ablation_rows)
dimension_ablation_results["rank_within_model"] = (
    dimension_ablation_results.groupby("model")["performance_loss"]
    .rank(ascending=False, method="min")
    .astype(int)
)
dimension_ablation_results = dimension_ablation_results.sort_values(
    ["model", "rank_within_model", "omitted_dimension"]
).reset_index(drop=True)
dimension_ablation_results.to_csv(
    OUTPUT_DIR / "dimension_ablation_results_v0.1.csv", index=False
)

display(dimension_ablation_results.pivot(
    index="omitted_dimension",
    columns="model",
    values="performance_loss",
).sort_values("Ordinal Logistic Regression", ascending=False).round(4))

model,Binary Logistic Regression,Ordinal Logistic Regression,XGBoost
omitted_dimension,,,
Teaching on my course,0.0175,0.0470,0.0175
Organisation and management,0.0052,0.0099,0.0049
Learning community,0.0013,0.0039,0.0018
Learning resources,0.0010,0.0029,0.0013
Student voice,0.0009,0.0024,0.0010
Academic support,0.0006,0.0021,0.0007
Learning opportunities,0.0009,0.0014,0.0012
Assessment and feedback,0.0008,0.0011,0.0009


## 24. Subgroup performance audit

Performance is reported separately for sex, age group, reported disability and ethnicity code. Groups with fewer than 100 test observations are suppressed. The audit compares dimension-only and dimension-plus-characteristic specifications for the primary ordinal model, binary logistic regression and XGBoost. For the combined specifications, percentile bootstrap intervals quantify test-sample uncertainty in QWK and ROC-AUC. These diagnostics describe subgroup performance; they do not establish algorithmic fairness or explain the causes of group differences.

In [30]:
SUBGROUP_CHARACTERISTICS = [
    "sex", "age_group", "disability_reported", "ethnicity_code"
]
SUBGROUP_MIN_GROUP_N = 100
SUBGROUP_BOOTSTRAP_REPS = 1000


def ordinal_subgroup_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    return {
        **evaluate_ordinal_predictions(y_true, y_pred),
        "actual_mean_q27": y_true.mean(),
        "predicted_mean_q27": y_pred.mean(),
        "mean_prediction_gap": y_pred.mean() - y_true.mean(),
        "actual_satisfied_rate": np.mean(y_true >= 4),
    }


def binary_subgroup_metrics(y_true, y_pred, y_prob):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    y_prob = np.asarray(y_prob, dtype=float)
    true_positive = np.sum((y_true == 1) & (y_pred == 1))
    false_positive = np.sum((y_true == 0) & (y_pred == 1))
    return {
        **evaluate_binary_predictions(y_true, y_pred, y_prob),
        "brier_score": brier_score_loss(y_true, y_prob),
        "actual_satisfied_rate": y_true.mean(),
        "predicted_satisfied_rate": y_pred.mean(),
        "mean_predicted_probability": y_prob.mean(),
        "calibration_gap": y_prob.mean() - y_true.mean(),
        "true_positive_rate": true_positive / np.sum(y_true == 1),
        "false_positive_rate": false_positive / np.sum(y_true == 0),
    }


def bootstrap_metric_interval(y_true, estimate_values, metric_name, rng):
    y_true = np.asarray(y_true)
    estimate_values = np.asarray(estimate_values)
    estimates = []
    for _ in range(SUBGROUP_BOOTSTRAP_REPS):
        sample = rng.integers(0, len(y_true), len(y_true))
        sample_true = y_true[sample]
        if metric_name == "roc_auc" and np.unique(sample_true).size < 2:
            continue
        if metric_name == "quadratic_weighted_kappa":
            estimate = cohen_kappa_score(
                sample_true, estimate_values[sample], weights="quadratic"
            )
        else:
            estimate = roc_auc_score(sample_true, estimate_values[sample])
        if np.isfinite(estimate):
            estimates.append(estimate)
    return np.quantile(estimates, [0.025, 0.975])


audit_predictions = []
for group in ["module_scores", "module_plus_demographic"]:
    scaled_test = scaled_design_matrices[group][1]
    tree_test = tree_design_matrices[group][1]
    ordinal_result = ordinal_artifacts[group]["result"]
    ordinal_probability = ordinal_result.model.predict(
        ordinal_result.params, exog=scaled_test
    )
    logistic = binary_logistic_artifacts[group]["classifier"]
    xgboost = binary_model_artifacts[("XGBoost", group)]["classifier"]
    audit_predictions.extend([
        {
            "task": "ordinal_1_to_5", "model": "Ordinal Logistic Regression",
            "feature_group": group,
            "prediction": np.asarray(ordinal_probability).argmax(axis=1) + 1,
            "probability": None,
        },
        {
            "task": "binary_satisfied", "model": "Binary Logistic Regression",
            "feature_group": group, "prediction": logistic.predict(scaled_test),
            "probability": logistic.predict_proba(scaled_test)[:, 1],
        },
        {
            "task": "binary_satisfied", "model": "XGBoost",
            "feature_group": group, "prediction": xgboost.predict(tree_test),
            "probability": xgboost.predict_proba(tree_test)[:, 1],
        },
    ])

subgroup_rows = []
bootstrap_rng = np.random.default_rng(RANDOM_STATE)
for artifact in audit_predictions:
    true_outcome = np.asarray(
        y_test_ord if artifact["task"] == "ordinal_1_to_5" else y_test_bin,
        dtype=int,
    )
    for characteristic in SUBGROUP_CHARACTERISTICS:
        values = test_df[characteristic].astype(str).to_numpy()
        for group_value in sorted(np.unique(values)):
            mask = values == group_value
            if mask.sum() < SUBGROUP_MIN_GROUP_N or np.unique(true_outcome[mask]).size < 2:
                continue
            group_true = true_outcome[mask]
            if artifact["task"] == "ordinal_1_to_5":
                group_prediction = artifact["prediction"][mask]
                metrics = ordinal_subgroup_metrics(group_true, group_prediction)
                if artifact["feature_group"] == "module_plus_demographic":
                    lower, upper = bootstrap_metric_interval(
                        group_true, group_prediction,
                        "quadratic_weighted_kappa", bootstrap_rng,
                    )
                    metrics.update({
                        "qwk_ci_lower": lower,
                        "qwk_ci_upper": upper,
                    })
            else:
                group_prediction = artifact["prediction"][mask]
                group_probability = artifact["probability"][mask]
                metrics = binary_subgroup_metrics(
                    group_true, group_prediction, group_probability
                )
                if artifact["feature_group"] == "module_plus_demographic":
                    lower, upper = bootstrap_metric_interval(
                        group_true, group_probability, "roc_auc", bootstrap_rng
                    )
                    metrics.update({
                        "roc_auc_ci_lower": lower,
                        "roc_auc_ci_upper": upper,
                    })
            subgroup_rows.append({
                "task": artifact["task"],
                "model": artifact["model"],
                "feature_group": artifact["feature_group"],
                "characteristic": characteristic,
                "group": group_value,
                "n": int(mask.sum()),
                **metrics,
            })

subgroup_performance_detailed = pd.DataFrame(subgroup_rows)
range_metrics = {
    "ordinal_1_to_5": [
        "quadratic_weighted_kappa", "mean_absolute_error", "mean_prediction_gap"
    ],
    "binary_satisfied": [
        "roc_auc", "balanced_accuracy", "macro_f1", "brier_score",
        "true_positive_rate", "false_positive_rate", "calibration_gap",
    ],
}
range_rows = []
for keys, rows in subgroup_performance_detailed.groupby(
    ["task", "model", "feature_group", "characteristic"]
):
    summary = {
        "task": keys[0], "model": keys[1], "feature_group": keys[2],
        "characteristic": keys[3], "groups_reported": len(rows),
        "minimum_group_n": rows["n"].min(),
    }
    for metric in range_metrics[keys[0]]:
        summary.update({
            f"{metric}_min": rows[metric].min(),
            f"{metric}_max": rows[metric].max(),
            f"{metric}_range": rows[metric].max() - rows[metric].min(),
        })
    range_rows.append(summary)

subgroup_performance_ranges = pd.DataFrame(range_rows).sort_values(
    ["task", "model", "feature_group", "characteristic"]
).reset_index(drop=True)
subgroup_performance_detailed.to_csv(
    OUTPUT_DIR / "subgroup_performance_detailed_v0.1.csv", index=False
)
subgroup_performance_ranges.to_csv(
    OUTPUT_DIR / "subgroup_performance_ranges_v0.1.csv", index=False
)

combined_ordinal = subgroup_performance_detailed.loc[
    (subgroup_performance_detailed["task"] == "ordinal_1_to_5")
    & (subgroup_performance_detailed["model"] == "Ordinal Logistic Regression")
    & (subgroup_performance_detailed["feature_group"] == "module_plus_demographic"),
    [
        "characteristic", "group", "n", "actual_satisfied_rate",
        "quadratic_weighted_kappa", "qwk_ci_lower", "qwk_ci_upper",
    ],
].copy()
combined_logistic = subgroup_performance_detailed.loc[
    (subgroup_performance_detailed["task"] == "binary_satisfied")
    & (subgroup_performance_detailed["model"] == "Binary Logistic Regression")
    & (subgroup_performance_detailed["feature_group"] == "module_plus_demographic"),
    ["characteristic", "group", "roc_auc", "roc_auc_ci_lower", "roc_auc_ci_upper"],
].rename(columns={
    "roc_auc": "logistic_roc_auc",
    "roc_auc_ci_lower": "logistic_roc_auc_ci_lower",
    "roc_auc_ci_upper": "logistic_roc_auc_ci_upper",
})
combined_xgboost = subgroup_performance_detailed.loc[
    (subgroup_performance_detailed["task"] == "binary_satisfied")
    & (subgroup_performance_detailed["model"] == "XGBoost")
    & (subgroup_performance_detailed["feature_group"] == "module_plus_demographic"),
    ["characteristic", "group", "roc_auc", "roc_auc_ci_lower", "roc_auc_ci_upper"],
].rename(columns={
    "roc_auc": "xgboost_roc_auc",
    "roc_auc_ci_lower": "xgboost_roc_auc_ci_lower",
    "roc_auc_ci_upper": "xgboost_roc_auc_ci_upper",
})
subgroup_performance_appendix = (
    combined_ordinal
    .merge(combined_logistic, on=["characteristic", "group"], how="left")
    .merge(combined_xgboost, on=["characteristic", "group"], how="left")
)
characteristic_order = {
    name: order for order, name in enumerate(SUBGROUP_CHARACTERISTICS)
}
subgroup_performance_appendix["characteristic_order"] = (
    subgroup_performance_appendix["characteristic"].map(characteristic_order)
)
subgroup_performance_appendix = subgroup_performance_appendix.sort_values(
    ["characteristic_order", "group"]
).drop(columns="characteristic_order").reset_index(drop=True)
subgroup_performance_appendix.to_csv(
    OUTPUT_DIR / "subgroup_performance_appendix_v0.1.csv", index=False
)

ordinal_columns = [
    "model", "feature_group", "characteristic", "groups_reported",
    "minimum_group_n", "quadratic_weighted_kappa_range",
    "mean_absolute_error_range", "mean_prediction_gap_range",
]
binary_columns = [
    "model", "feature_group", "characteristic", "groups_reported",
    "minimum_group_n", "roc_auc_range", "balanced_accuracy_range",
    "true_positive_rate_range", "false_positive_rate_range",
    "calibration_gap_range",
]
display(subgroup_performance_ranges.loc[
    subgroup_performance_ranges["task"] == "ordinal_1_to_5", ordinal_columns
].round(4))
display(subgroup_performance_ranges.loc[
    subgroup_performance_ranges["task"] == "binary_satisfied", binary_columns
].round(4))
display(subgroup_performance_appendix.round(4))

,model,feature_group,characteristic,groups_reported,minimum_group_n,quadratic_weighted_kappa_range,mean_absolute_error_range,mean_prediction_gap_range
16,Ordinal Logistic Regression,module_plus_demographic,age_group,4,310,0.0563,0.0477,0.0766
17,Ordinal Logistic Regression,module_plus_demographic,disability_reported,2,1937,0.0273,0.0035,0.0062
18,Ordinal Logistic Regression,module_plus_demographic,ethnicity_code,15,101,0.2080,0.1923,0.1963
19,Ordinal Logistic Regression,module_plus_demographic,sex,2,1723,0.0234,0.0019,0.0073
20,Ordinal Logistic Regression,module_scores,age_group,4,310,0.0571,0.0609,0.0379
21,Ordinal Logistic Regression,module_scores,disability_reported,2,1937,0.0266,0.0045,0.0097
22,Ordinal Logistic Regression,module_scores,ethnicity_code,15,101,0.2007,0.1743,0.1669
23,Ordinal Logistic Regression,module_scores,sex,2,1723,0.0188,0.0039,0.0020


,model,feature_group,characteristic,groups_reported,minimum_group_n,roc_auc_range,balanced_accuracy_range,true_positive_rate_range,false_positive_rate_range,calibration_gap_range
0,Binary Logistic Regression,module_plus_demographic,age_group,4,310,0.0263,0.0494,0.0495,0.1385,0.0238
1,Binary Logistic Regression,module_plus_demographic,disability_reported,2,1937,0.0064,0.0177,0.0008,0.0346,0.0028
2,Binary Logistic Regression,module_plus_demographic,ethnicity_code,15,101,0.0782,0.0949,0.0729,0.1908,0.0740
3,Binary Logistic Regression,module_plus_demographic,sex,2,1723,0.0018,0.0061,0.0031,0.0153,0.0027
4,Binary Logistic Regression,module_scores,age_group,4,310,0.0282,0.0371,0.0487,0.1228,0.0182
5,Binary Logistic Regression,module_scores,disability_reported,2,1937,0.0069,0.0079,0.0016,0.0142,0.0055
6,Binary Logistic Regression,module_scores,ethnicity_code,15,101,0.0736,0.1144,0.0631,0.2254,0.0807
7,Binary Logistic Regression,module_scores,sex,2,1723,0.0022,0.0139,0.0043,0.0320,0.0032
8,XGBoost,module_plus_demographic,age_group,4,310,0.0242,0.0538,0.0410,0.1358,0.0162
9,XGBoost,module_plus_demographic,disability_reported,2,1937,0.0058,0.0118,0.0052,0.0184,0.0055


,characteristic,group,n,actual_satisfied_rate,quadratic_weighted_kappa,qwk_ci_lower,qwk_ci_upper,logistic_roc_auc,logistic_roc_auc_ci_lower,logistic_roc_auc_ci_upper,xgboost_roc_auc,xgboost_roc_auc_ci_lower,xgboost_roc_auc_ci_upper
0,sex,F,1723,0.7487,0.7718,0.7456,0.7959,0.9399,0.9270,0.9511,0.9424,0.9299,0.9539
1,sex,M,8404,0.7506,0.7952,0.7841,0.8055,0.9382,0.9326,0.9437,0.9389,0.9334,0.9439
2,age_group,21_25,1285,0.7572,0.8244,0.7989,0.8469,0.9543,0.9415,0.9661,0.9549,0.9420,0.9665
3,age_group,26_30,310,0.6935,0.8377,0.7834,0.8817,0.9427,0.9138,0.9648,0.9441,0.9124,0.9691
4,age_group,31+,350,0.7514,0.8318,0.7855,0.8685,0.9609,0.9389,0.9771,0.9600,0.9377,0.9787
5,age_group,U21,8182,0.7513,0.7814,0.7709,0.7924,0.9346,0.9288,0.9407,0.9358,0.9301,0.9412
6,disability_reported,0,8190,0.7505,0.7857,0.7745,0.7970,0.9374,0.9316,0.9430,0.9385,0.9329,0.9443
7,disability_reported,1,1937,0.7491,0.8129,0.7918,0.8326,0.9437,0.9330,0.9533,0.9443,0.9334,0.9543
8,ethnicity_code,10,6156,0.7529,0.7994,0.7879,0.8111,0.9422,0.9365,0.9481,0.9425,0.9361,0.9485
9,ethnicity_code,13,240,0.8125,0.7608,0.6787,0.8248,0.9427,0.9046,0.9719,0.9371,0.8962,0.9707


## 25. Extended validation summary

The following compact output identifies the main temporal transport, ablation and subgroup-performance results. Interpretation should consider absolute performance, bootstrap uncertainty, group sample sizes and the observational design together.

In [31]:
temporal_primary = temporal_validation_results.loc[
    (temporal_validation_results["task"] == "ordinal_1_to_5")
    & (temporal_validation_results["model"] == "Ordinal Logistic Regression")
    & (temporal_validation_results["feature_group"] == "module_scores")
].iloc[0]
temporal_best_binary = temporal_validation_results.loc[
    temporal_validation_results["task"] == "binary_satisfied"
].sort_values("roc_auc", ascending=False).iloc[0]
ordinal_ablation = dimension_ablation_results.loc[
    dimension_ablation_results["model"] == "Ordinal Logistic Regression"
].sort_values("performance_loss", ascending=False)
largest_ordinal_range = subgroup_performance_ranges.loc[
    (subgroup_performance_ranges["task"] == "ordinal_1_to_5")
    & (subgroup_performance_ranges["feature_group"] == "module_plus_demographic")
].sort_values("quadratic_weighted_kappa_range", ascending=False).iloc[0]
largest_binary_range = subgroup_performance_ranges.loc[
    (subgroup_performance_ranges["task"] == "binary_satisfied")
    & (subgroup_performance_ranges["feature_group"] == "module_plus_demographic")
].sort_values("roc_auc_range", ascending=False).iloc[0]

print(
    f"Later-year transport performance for the primary ordinal model: QWK "
    f"{temporal_primary['quadratic_weighted_kappa']:.3f}, MAE "
    f"{temporal_primary['mean_absolute_error']:.3f}."
)
print(
    f"Best later-year binary ROC-AUC: {temporal_best_binary['roc_auc']:.3f} "
    f"({temporal_best_binary['model']}, {temporal_best_binary['feature_group']})."
)
print(
    f"Largest ordinal ablation loss: {ordinal_ablation.iloc[0]['omitted_dimension']} "
    f"(QWK loss {ordinal_ablation.iloc[0]['performance_loss']:.3f})."
)
print(
    f"Largest observed ordinal subgroup QWK range: "
    f"{largest_ordinal_range['characteristic']} "
    f"({largest_ordinal_range['quadratic_weighted_kappa_range']:.3f})."
)
print(
    f"Largest observed binary subgroup ROC-AUC range: "
    f"{largest_binary_range['characteristic']} "
    f"({largest_binary_range['roc_auc_range']:.3f})."
)

Later-year transport performance for the primary ordinal model: QWK 0.794, MAE 0.390.
Best later-year binary ROC-AUC: 0.935 (XGBoost, module_plus_demographic_no_year).
Largest ordinal ablation loss: Teaching on my course (QWK loss 0.047).
Largest observed ordinal subgroup QWK range: ethnicity_code (0.208).
Largest observed binary subgroup ROC-AUC range: ethnicity_code (0.079).


## 26. Midpoint-coding sensitivity analysis

The primary analysis treats Q1–Q26 responses coded 0 (Not applicable) as missing when constructing the eight NSS dimension means. This sensitivity analysis instead codes each 0 response as the Likert midpoint 3 (Neither agree nor disagree). Q27 responses coded 0 remain excluded because the outcome is not observed. The comparison tests whether model performance, primary ordinal coefficients and factor ranking depend materially on the treatment of Not applicable responses. Midpoint coding is an alternative assumption rather than a preferred interpretation: Not applicable does not necessarily mean neutral.

In [32]:
MIDPOINT_VALUE = 3


def add_midpoint_module_scores(frame):
    scored = frame.copy()
    for module in module_definitions:
        scored[module["module_score_col"]] = (
            scored[module["items"]].replace(0, MIDPOINT_VALUE).mean(axis=1)
        )
    return scored


def midpoint_comparison_row(task, model, feature_group, metrics):
    original = combined_results.loc[
        (combined_results["task"] == task)
        & (combined_results["model"] == model)
        & (combined_results["feature_group"] == feature_group)
    ].iloc[0]
    row = {
        "task": task,
        "model": model,
        "feature_group": feature_group,
        "primary_coding": "0 treated as missing",
        "sensitivity_coding": "0 coded as midpoint 3",
    }
    for metric in [
        "accuracy", "macro_f1", "quadratic_weighted_kappa",
        "mean_absolute_error", "balanced_accuracy", "roc_auc",
    ]:
        original_value = original.get(metric, np.nan)
        midpoint_value = metrics.get(metric, np.nan)
        row[f"original_{metric}"] = original_value
        row[f"midpoint_{metric}"] = midpoint_value
        row[f"change_{metric}"] = midpoint_value - original_value
    return row


midpoint_modules = clean_demographics(add_midpoint_module_scores(df_raw))
midpoint_modules = midpoint_modules.loc[midpoint_modules[target_col] != 0].copy()
midpoint_train = midpoint_modules.loc[train_idx]
midpoint_test = midpoint_modules.loc[test_idx]

midpoint_scaled = {
    group: build_design_matrices(
        midpoint_train, midpoint_test, group, scale_numeric=True
    )
    for group in ["module_scores", "module_plus_demographic"]
}
midpoint_tree = {
    group: build_design_matrices(
        midpoint_train, midpoint_test, group, scale_numeric=False
    )
    for group in ["module_scores", "module_plus_demographic"]
}

midpoint_rows = []
midpoint_ordinal_artifacts = {}
for group in ["module_scores", "module_plus_demographic"]:
    scaled_train, scaled_test, feature_names, preprocessor = midpoint_scaled[group]
    ordinal_model = OrderedModel(y_train_ord, scaled_train, distr="logit")
    ordinal_result = ordinal_model.fit(
        method="lbfgs", maxiter=ORDINAL_MAXITER, disp=False
    )
    ordinal_probability = ordinal_result.model.predict(
        ordinal_result.params, exog=scaled_test
    )
    ordinal_metrics = evaluate_ordinal_predictions(
        y_test_ord, np.asarray(ordinal_probability).argmax(axis=1) + 1
    )
    midpoint_rows.append(midpoint_comparison_row(
        "ordinal_1_to_5", "Ordinal Logistic Regression", group, ordinal_metrics
    ))
    midpoint_ordinal_artifacts[group] = {
        "result": ordinal_result,
        "feature_names": feature_names,
        "preprocessor": preprocessor,
    }

    logistic = LogisticRegression(max_iter=3000, solver="lbfgs")
    logistic.fit(scaled_train, y_train_bin)
    logistic_prediction = logistic.predict(scaled_test)
    logistic_probability = logistic.predict_proba(scaled_test)[:, 1]
    midpoint_rows.append(midpoint_comparison_row(
        "binary_satisfied", "Binary Logistic Regression", group,
        evaluate_binary_predictions(
            y_test_bin, logistic_prediction, logistic_probability
        ),
    ))

    tree_train, tree_test, _, _ = midpoint_tree[group]
    xgboost = XGBClassifier(
        n_estimators=250, max_depth=3, learning_rate=0.05,
        subsample=0.9, colsample_bytree=0.9,
        objective="binary:logistic", eval_metric="logloss",
        random_state=RANDOM_STATE, n_jobs=-1, tree_method="hist",
    )
    xgboost.fit(tree_train, y_train_bin)
    xgboost_prediction = xgboost.predict(tree_test)
    xgboost_probability = xgboost.predict_proba(tree_test)[:, 1]
    midpoint_rows.append(midpoint_comparison_row(
        "binary_satisfied", "XGBoost", group,
        evaluate_binary_predictions(
            y_test_bin, xgboost_prediction, xgboost_probability
        ),
    ))

midpoint_coding_performance = pd.DataFrame(midpoint_rows).sort_values(
    ["task", "model", "feature_group"]
).reset_index(drop=True)
midpoint_coding_performance.to_csv(
    OUTPUT_DIR / "midpoint_coding_performance_v0.1.csv", index=False
)

original_primary = ordinal_artifacts["module_scores"]
midpoint_primary = midpoint_ordinal_artifacts["module_scores"]
feature_names = midpoint_primary["feature_names"]
n_features = len(feature_names)
original_coefficients = np.asarray(
    original_primary["result"].params[:n_features], dtype=float
)
original_standard_errors = np.asarray(
    original_primary["result"].bse[:n_features], dtype=float
)
midpoint_coefficients = np.asarray(
    midpoint_primary["result"].params[:n_features], dtype=float
)
midpoint_standard_errors = np.asarray(
    midpoint_primary["result"].bse[:n_features], dtype=float
)

midpoint_coding_coefficients = pd.DataFrame({
    "feature": feature_names,
    "nss_module": [module_name_by_col[feature] for feature in feature_names],
    "question_range": [module_range_by_col[feature] for feature in feature_names],
    "original_coefficient": original_coefficients,
    "original_ci_lower": original_coefficients - 1.96 * original_standard_errors,
    "original_ci_upper": original_coefficients + 1.96 * original_standard_errors,
    "midpoint_coefficient": midpoint_coefficients,
    "midpoint_ci_lower": midpoint_coefficients - 1.96 * midpoint_standard_errors,
    "midpoint_ci_upper": midpoint_coefficients + 1.96 * midpoint_standard_errors,
})
midpoint_coding_coefficients["coefficient_change"] = (
    midpoint_coding_coefficients["midpoint_coefficient"]
    - midpoint_coding_coefficients["original_coefficient"]
)
midpoint_coding_coefficients["original_odds_ratio"] = np.exp(
    midpoint_coding_coefficients["original_coefficient"]
)
midpoint_coding_coefficients["midpoint_odds_ratio"] = np.exp(
    midpoint_coding_coefficients["midpoint_coefficient"]
)
for coding in ["original", "midpoint"]:
    midpoint_coding_coefficients[f"{coding}_rank"] = (
        midpoint_coding_coefficients[f"{coding}_coefficient"].abs()
        .rank(ascending=False, method="min").astype(int)
    )
midpoint_coding_coefficients["rank_change"] = (
    midpoint_coding_coefficients["midpoint_rank"]
    - midpoint_coding_coefficients["original_rank"]
)
midpoint_coding_coefficients = midpoint_coding_coefficients.sort_values(
    "midpoint_rank"
).reset_index(drop=True)
midpoint_coding_coefficients.to_csv(
    OUTPUT_DIR / "midpoint_coding_ordinal_coefficients_v0.1.csv", index=False
)

module_rows = []
for module in module_definitions:
    items = module["items"]
    feature = module["module_score_col"]
    module_rows.append({
        "nss_module": module["nss_module"],
        "question_range": module["question_range"],
        "not_applicable_responses": int((df_raw[items] == 0).sum().sum()),
        "rows_affected": int((df_raw[items] == 0).any(axis=1).sum()),
        "original_mean": df_modules[feature].mean(),
        "midpoint_coded_mean": midpoint_modules[feature].mean(),
        "mean_change": midpoint_modules[feature].mean() - df_modules[feature].mean(),
    })
midpoint_coding_module_summary = pd.DataFrame(module_rows)
midpoint_coding_module_summary.to_csv(
    OUTPUT_DIR / "midpoint_coding_module_summary_v0.1.csv", index=False
)

rank_correlation = midpoint_coding_coefficients[
    ["original_rank", "midpoint_rank"]
].corr(method="spearman").iloc[0, 1]
primary_performance = midpoint_coding_performance.loc[
    (midpoint_coding_performance["task"] == "ordinal_1_to_5")
    & (midpoint_coding_performance["feature_group"] == "module_scores")
].iloc[0]

display(midpoint_coding_module_summary.round(4))
display(midpoint_coding_performance.round(4))
display(midpoint_coding_coefficients.round(4))
print(
    f"Midpoint coding changes primary ordinal QWK from "
    f"{primary_performance['original_quadratic_weighted_kappa']:.3f} to "
    f"{primary_performance['midpoint_quadratic_weighted_kappa']:.3f}."
)
print(
    f"The Spearman correlation between original and midpoint-coded factor ranks "
    f"is {rank_correlation:.3f}."
)

,nss_module,question_range,not_applicable_responses,rows_affected,original_mean,midpoint_coded_mean,mean_change
0,Teaching on my course,Q1-Q4,172,155,3.9595,3.9594,-0.0002
1,Learning opportunities,Q5-Q7,234,218,4.0111,4.0104,-0.0007
2,Assessment and feedback,Q8-Q11,312,249,3.7130,3.7127,-0.0003
3,Academic support,Q12-Q14,1502,1337,3.9652,3.9566,-0.0086
4,Organisation and management,Q15-Q17,679,650,3.7926,3.7902,-0.0024
5,Learning resources,Q18-Q20,3629,3126,4.0885,4.0662,-0.0224
6,Learning community,Q21-Q22,667,602,3.9045,3.9008,-0.0036
7,Student voice,Q23-Q26,4519,3556,3.7561,3.7431,-0.0131


,task,model,feature_group,primary_coding,sensitivity_coding,original_accuracy,midpoint_accuracy,change_accuracy,original_macro_f1,midpoint_macro_f1,...,change_quadratic_weighted_kappa,original_mean_absolute_error,midpoint_mean_absolute_error,change_mean_absolute_error,original_balanced_accuracy,midpoint_balanced_accuracy,change_balanced_accuracy,original_roc_auc,midpoint_roc_auc,change_roc_auc
0,binary_satisfied,Binary Logistic Regression,module_plus_demographic,0 treated as missing,0 coded as midpoint 3,0.8906,0.8905,-0.0001,0.8449,0.8450,...,NaN,NaN,NaN,NaN,0.8258,0.8264,0.0006,0.9386,0.9388,0.0003
1,binary_satisfied,Binary Logistic Regression,module_scores,0 treated as missing,0 coded as midpoint 3,0.8896,0.8903,0.0007,0.8437,0.8446,...,NaN,NaN,NaN,NaN,0.8251,0.8257,0.0006,0.9384,0.9387,0.0003
2,binary_satisfied,XGBoost,module_plus_demographic,0 treated as missing,0 coded as midpoint 3,0.8891,0.8892,0.0001,0.8445,0.8443,...,NaN,NaN,NaN,NaN,0.8285,0.8278,-0.0007,0.9396,0.9397,0.0001
3,binary_satisfied,XGBoost,module_scores,0 treated as missing,0 coded as midpoint 3,0.8876,0.8880,0.0004,0.8425,0.8431,...,NaN,NaN,NaN,NaN,0.8268,0.8275,0.0007,0.9392,0.9394,0.0003
4,ordinal_1_to_5,Ordinal Logistic Regression,module_plus_demographic,0 treated as missing,0 coded as midpoint 3,0.6627,0.6622,-0.0005,0.5683,0.5669,...,0.0003,0.3771,0.3772,0.0001,NaN,NaN,NaN,NaN,NaN,NaN
5,ordinal_1_to_5,Ordinal Logistic Regression,module_scores,0 treated as missing,0 coded as midpoint 3,0.6638,0.6627,-0.0011,0.5673,0.5658,...,-0.0002,0.3758,0.3767,0.0009,NaN,NaN,NaN,NaN,NaN,NaN


,feature,nss_module,question_range,original_coefficient,original_ci_lower,original_ci_upper,midpoint_coefficient,midpoint_ci_lower,midpoint_ci_upper,coefficient_change,original_odds_ratio,midpoint_odds_ratio,original_rank,midpoint_rank,rank_change
0,module_teaching_on_my_course,Teaching on my course,Q1-Q4,1.2944,1.2572,1.3315,1.2914,1.2542,1.3286,-0.0030,3.6486,3.6378,1,1,0
1,module_organisation_and_management,Organisation and management,Q15-Q17,0.6199,0.5878,0.6520,0.6172,0.5851,0.6494,-0.0026,1.8587,1.8538,2,2,0
2,module_student_voice,Student voice,Q23-Q26,0.3776,0.3434,0.4118,0.3873,0.3529,0.4217,0.0097,1.4588,1.4730,4,3,-1
3,module_learning_opportunities,Learning opportunities,Q5-Q7,0.3836,0.3512,0.4161,0.3812,0.3487,0.4136,-0.0024,1.4676,1.4640,3,4,1
4,module_academic_support,Academic support,Q12-Q14,0.2940,0.2616,0.3264,0.2904,0.2580,0.3229,-0.0036,1.3418,1.3370,5,5,0
5,module_learning_resources,Learning resources,Q18-Q20,0.2501,0.2238,0.2765,0.2529,0.2264,0.2794,0.0028,1.2842,1.2878,6,6,0
6,module_learning_community,Learning community,Q21-Q22,0.2496,0.2203,0.2790,0.2467,0.2173,0.2762,-0.0029,1.2836,1.2798,7,7,0
7,module_assessment_and_feedback,Assessment and feedback,Q8-Q11,0.2094,0.1790,0.2398,0.2104,0.1801,0.2408,0.0010,1.2330,1.2342,8,8,0


Midpoint coding changes primary ordinal QWK from 0.792 to 0.792.
The Spearman correlation between original and midpoint-coded factor ranks is 0.976.
